# Geneformer In-Silico Perturbation (ISP) Pipeline
### Based on Liu et al. — *Evaluating Foundation Models for In-Silico Perturbation*

---

**Pipeline overview:**
1. Install dependencies
2. Load and QC `.h5ad` files
3. Auto-annotate cell types (CellTypist)
4. Format AnnData for Geneformer (raw counts → Ensembl IDs → tokenization)
5. Generate baseline embeddings (control & target states)
6. Run separation test
7. Select candidate perturbation genes
8. Run in-silico perturbation (down-regulation / deletion / activation)
9. Compute cosine shift scores
10. Build random baseline & statistical testing
11. Rank genes and output results table
12. Biological validation (pathway enrichment)

---


## 0.Install Dependencies

In [ ]:
# Core single-cell stack
!pip install -q anndata scanpy scipy pandas numpy matplotlib seaborn

# Geneformer (from HuggingFace / Theodoris lab)
!pip install -q transformers datasets
!pip install -q pyarrow

# loompy for writing loom files (anndata.write_loom is deprecated)
!pip install -q loompy

# Cell type annotation
!pip install -q celltypist

# Gene ID mapping
!pip install -q mygene

# Pathway enrichment
!pip install -q gseapy

# Stats
!pip install -q statsmodels scikit-learn

# Clone Geneformer repo (for tokenizer + ISP utilities)
import os
if not os.path.exists('/content/Geneformer'):
    !git clone https://huggingface.co/ctheodoris/Geneformer /content/Geneformer

# Install Geneformer package (separate step to avoid %cd issues)
!pip install -q -e /content/Geneformer

# Add to path
import sys
if '/content/Geneformer' not in sys.path:
    sys.path.insert(0, '/content/Geneformer')

print('✅ All dependencies installed.')

In [ ]:
!pip install "transformers==4.57.1"
!pip install .

In [ ]:
# ensure datasets updated in Colab environment
!pip install gcsfs==2025.3.0
!pip install datasets==3.6.0

## 1.Configuration — Set Parameters Here

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys
if '/content/Geneformer' not in sys.path:
    sys.path.insert(0, '/content/Geneformer')

H5AD_PATHS = [
    '/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE159977.h5ad',
    '/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE185477.h5ad',
    '/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE189600.h5ad',
    '/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE190487.h5ad',
    '/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE192740.h5ad',
    '/content/drive/MyDrive/scFM perturbation project/input_scFM/GSE202379.h5ad'
]

CONDITION_COL  = None
CONTROL_LABEL  = 'Healthy'
TARGET_LABEL   = 'MASH'
ALT_STATES = ['MASLD']
CELLTYPE_COL   = None
FOCUS_CELLTYPES = ['CD16- NK cells', 'Tem/Trm cytotoxic T cells', 'CD16+ NK cells', 'MAIT cells',
                   'Tem/Effector helper T cells', 'NK cells', 'Tem/Temra cytotoxic T cells',
                   'Tcm/Naive helper T cells', 'Naive B cells', 'CRTAM+ gamma-delta T cells',
                   'Hepatocytes', 'T cells', 'Endothelial cells', 'Fibroblasts', 'Macrophages',
                   'Cholangiocytes', 'Mono+mono derived cells', 'B cells', 'Plasma cells',
                   'Memory B cells', 'DC2', 'Classical monocytes', 'pDC',
                   'Intestinal macrophages', 'Non-classical monocytes', 'Resident NK',
                   'Circulating NK/NKT', 'Neutrophils']

# ── DONOR / PATIENT COLUMN ─────────────────────────────────────────────
# Set to the obs column containing donor/patient IDs for patient-aware sampling.
# If None, the pipeline will attempt auto-detection from common column names.
# Set to False to disable patient-aware sampling entirely (not recommended).
DONOR_COL = 'patient_id'  # e.g. 'donor', 'patient_id', 'sample_id'

# ISP patient-aware sampling budget (cells per cell-type/condition per ISP run)
MAX_ISP_CELLS_TOTAL     = 500   # max cells fed to ISP per cell-type
MIN_CELLS_PER_DONOR_ISP = 5     # donors with fewer cells are excluded from ISP sampling

# ── SAVED EMBEDDINGS (optional) ────────────────────────────────────────
# If ran Step 6 and saved embeddings, point SAVED_EMBEDDINGS_PATH
# to the .parquet or .csv file to skip re-running the EmbExtractor.
# Set to None to always re-extract embeddings.
SAVED_EMBEDDINGS_PATH = '/content/drive/MyDrive/scFM perturbation project/geneformer_embs_new.parquet'

GENE_ID_TYPE   = 'symbol'
SPECIES        = 'human'
PERTURB_MODE   = 'down'
N_CANDIDATE_GENES = 200
MIN_CELLS_PER_STATE = 100
OUTPUT_DIR = '/content/geneformer_isp_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✅ Configuration set.')
print(f'   Control        : {CONTROL_LABEL}')
print(f'   Target         : {TARGET_LABEL}')
print(f'   Perturb        : {PERTURB_MODE}-regulation')
print(f'   Donor col      : {DONOR_COL}')
print(f'   Saved embeddings: {SAVED_EMBEDDINGS_PATH}')


In [ ]:
# ── Auto-detect donor column from common column names ──────────────────
_DONOR_CANDIDATES = [
    'donor', 'donor_id', 'patient', 'patient_id', 'sample', 'sample_id',
    'subject', 'subject_id', 'individual', 'participant', 'Donor', 'Patient',
    'SampleID', 'orig.ident', 'batch',
]

def detect_donor_col(adata, override=None):
    """Return the first matching donor column found in adata.obs."""
    if override is not None and override is not False:
        if override in adata.obs.columns:
            return override
        print(f'   ⚠️  DONOR_COL="{override}" not found in obs.')
        return None
    if override is False:
        return None  # explicitly disabled
    for c in _DONOR_CANDIDATES:
        if c in adata.obs.columns:
            print(f'   Auto-detected donor column: "{c}"')
            return c
    print('   ⚠️  No donor column detected — patient-aware sampling will fall back to random.')
    return None

print('✅ Donor detection helper ready.')


## 2. Load & QC AnnData Objects

In [ ]:
import anndata as ad
import scanpy as sc
import numpy as np
import pandas as pd
import scipy.sparse as sp
import warnings
warnings.filterwarnings('ignore')
sc.settings.verbosity = 1

def detect_condition_col(adata):
    #candidates = ['condition','disease','group','diagnosis','phenotype','Status','status','Disease','Condition','sample_type','broad_condition','disease_state']
    candidate = ['broad_condition']
    for c in candidate:
        if c in adata.obs.columns:
            print(f'   Auto-detected condition column: "{c}"')
            print(f'   Values: {adata.obs[c].unique().tolist()}')
            return c
    return None

def load_and_qc(path, condition_col_override=None):
    print(f'\n📂 Loading: {path}')
    adata = sc.read_h5ad(path)
    print(f'   Shape  : {adata.shape[0]:,} cells × {adata.shape[1]:,} genes')
    print(f'   obs    : {list(adata.obs.columns)}')
    if adata.var_names.duplicated().any():
        print('   ⚠️  Duplicate gene names detected — making unique.')
        adata.var_names_make_unique()
    if 'counts' not in adata.layers:
        X = adata.X
        if sp.issparse(X):
            vals = X.data[:1000] if X.nnz > 1000 else X.data
        else:
            vals = np.asarray(X).flat[:1000]
        if np.allclose(vals, np.round(vals), atol=1e-3):
            print('   ✅ .X appears to be raw counts — copying to layers["counts"].')
            adata.layers['counts'] = adata.X.copy()
        else:
            print('   ⚠️  .X may be normalized. Checking adata.raw...')
            if adata.raw is not None:
                print('   ✅ Found adata.raw — using as raw counts.')
                raw_adata = adata.raw.to_adata()
                adata.layers['counts'] = raw_adata[:, adata.var_names].X.copy()
            else:
                print('   ❌ Cannot confirm raw counts. Proceeding with .X — verify manually!')
                adata.layers['counts'] = adata.X.copy()
    mito_prefix = 'MT-' if SPECIES == 'human' else 'mt-'
    adata.var['mt'] = adata.var_names.str.startswith(mito_prefix)
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    n_before = adata.n_obs
    adata = adata[adata.obs['n_genes_by_counts'] > 200].copy()
    adata = adata[adata.obs['pct_counts_mt'] < 20].copy()
    sc.pp.filter_genes(adata, min_cells=10)
    n_after = adata.n_obs
    print(f'   QC: {n_before:,} → {n_after:,} cells retained ({n_before - n_after:,} removed)')
    global CONDITION_COL
    if condition_col_override:
        adata.obs['_condition'] = adata.obs[condition_col_override].astype(str)
    elif CONDITION_COL and CONDITION_COL in adata.obs.columns:
        adata.obs['_condition'] = adata.obs[CONDITION_COL].astype(str)
    else:
        detected = detect_condition_col(adata)
        if detected:
            CONDITION_COL = detected
            adata.obs['_condition'] = adata.obs[detected].astype(str)
        else:
            print('   ❌ Could not detect condition column. Set CONDITION_COL manually.')
            adata.obs['_condition'] = 'unknown'
    print(f'   Conditions: {adata.obs["_condition"].value_counts().to_dict()}')
    return adata

adatas = []
for path in H5AD_PATHS:
    if os.path.exists(path):
        adatas.append(load_and_qc(path))
    else:
        print(f'⚠️  File not found: {path}')
if not adatas:
    raise FileNotFoundError('No .h5ad files loaded. Check H5AD_PATHS.')
print(f'\n✅ Loaded {len(adatas)} dataset(s).')

## 3. Cell Type Annotation
Uses **CellTypist** (pretrained on 36 human tissues) if cell type labels are not already present.

In [ ]:
for adata in adatas:
      if adata.obs.index.duplicated().any():
        print(adata)
        print(f"Duplicate cells before: {adata.obs.index.duplicated().sum()}")
        adata = adata[~adata.obs.index.duplicated(keep='first')]
        print(f"Duplicate cells after: {adata.obs.index.duplicated().sum()}")

In [ ]:
import celltypist
from celltypist import models
import scipy.sparse as sp

liver_gses = ['GSE212837', 'GSE189600', 'GSE174748', 'GSE192740', 'GSE185477', 'GSE202379']
immune_gses = ['GSE270488', 'GSE159977', 'GSE190487', 'GSE192740']

def get_celltypist_model_for_file(filename):
    """Return the appropriate CellTypist model name based on GSE in filename."""
    if filename is None:
        return 'Immune_All_Low.pkl'
    for gse in liver_gses:
        if gse in filename:
            return 'Healthy_Human_Liver.pkl'
    for gse in immune_gses:
        if gse in filename:
            return 'Immune_All_Low.pkl'
    return 'Immune_All_Low.pkl'  # default if no GSE matches

def annotate_cell_types(adata, use_celltypist=True, filename=None):
    global CELLTYPE_COL
    ct_candidates = ['cell_type','celltype','CellType','cell_ontology_class','Celltype','leiden_celltypes','anno','annotation','cluster_annotation','broad_celltypes']
    if CELLTYPE_COL and CELLTYPE_COL in adata.obs.columns:
        adata.obs['_cell_type'] = adata.obs[CELLTYPE_COL].astype(str)
        print(f'   Using existing cell type column: "{CELLTYPE_COL}"')
        print(f'   Types: {adata.obs["_cell_type"].value_counts().head(10).to_dict()}')
        return adata
    for c in ct_candidates:
        if c in adata.obs.columns:
            CELLTYPE_COL = c
            adata.obs['_cell_type'] = adata.obs[c].astype(str)
            print(f'   Auto-detected cell type column: "{c}"')
            print(f'   Types: {adata.obs["_cell_type"].value_counts().head(10).to_dict()}')
            return adata
    print('   No cell type labels found — running CellTypist annotation...')
    model_name = get_celltypist_model_for_file(filename)
    adata_ct = adata.copy()
    sc.pp.normalize_total(adata_ct, target_sum=1e4)
    sc.pp.log1p(adata_ct)
    try:
        model = models.Model.load(model=model_name)
        try:
            predictions = celltypist.annotate(adata_ct, model=model, majority_voting=True)
            adata.obs['_cell_type'] = predictions.predicted_labels['majority_voting'].values
        except Exception:
            # majority_voting fails on some datasets due to over-clustering mismatch
            # fall back to per-cell predictions without voting
            predictions = celltypist.annotate(adata_ct, model=model, majority_voting=False)
            adata.obs['_cell_type'] = predictions.predicted_labels['predicted_labels'].values
            print(f'   ⚠️  Majority voting failed, using per-cell predictions instead.')
        print(f'   ✅ CellTypist annotation complete ({model_name}).')
    except Exception as e:
        adata.obs['_cell_type'] = 'Unknown'
        print(f'   ❌ CellTypist failed ({e}). All cells labelled "Unknown".')
    print(f'   Types: {adata.obs["_cell_type"].value_counts().head(10).to_dict()}')
    return adata

for i, adata in enumerate(adatas):
    print(f'\n--- Dataset {i+1} ---')
    adatas[i] = annotate_cell_types(adata, filename=H5AD_PATHS[i])
print('\n✅ Cell type annotation complete.')

## 4. Map Gene Symbols → Ensembl IDs (Geneformer Vocabulary)

In [ ]:
import mygene
mg = mygene.MyGeneInfo()

def map_genes_to_ensembl(adata, species='human'):
    gene_names = adata.var_names.tolist()
    species_str = 'human' if species == 'human' else 'mouse'
    print(f'   Querying MyGene.info for {len(gene_names):,} genes...')
    results = mg.querymany(gene_names, scopes='symbol' if GENE_ID_TYPE=='symbol' else 'ensembl.gene', fields='ensembl.gene', species=species_str, returnall=False, as_dataframe=True, verbose=False)
    def extract_ensembl(row):
        val = row.get('ensembl.gene', np.nan)
        if isinstance(val, list): return val[0]
        return val
    results['ensembl_id'] = results.apply(extract_ensembl, axis=1)
    gene_map = results['ensembl_id'].dropna().to_dict()
    adata.var['ensembl_id'] = adata.var_names.map(gene_map)
    n_mapped = adata.var['ensembl_id'].notna().sum()
    print(f'   Mapped {n_mapped:,}/{len(gene_names):,} genes to Ensembl IDs.')
    adata = adata[:, adata.var['ensembl_id'].notna()].copy()
    print(f'   Final gene count: {adata.n_vars:,}')
    return adata

for i, adata in enumerate(adatas):
    print(f'\n--- Dataset {i+1} ---')
    adatas[i] = map_genes_to_ensembl(adata, species=SPECIES)
print('\n✅ Gene mapping complete.')

## 5. Tokenize Cells for Geneformer

In [ ]:
import sys
# Make sure Geneformer path is at the beginning of sys.path to prioritize its modules
if '/content/Geneformer' in sys.path:
    sys.path.remove('/content/Geneformer')
sys.path.insert(0, '/content/Geneformer')

import os, numpy as np, scipy.sparse as sp, loompy

# Now import TranscriptomeTokenizer after ensuring compatible transformers version
from geneformer import TranscriptomeTokenizer

TOKENIZED_DIR = os.path.join(OUTPUT_DIR, 'tokenized')
os.makedirs(TOKENIZED_DIR, exist_ok=True)

def prepare_for_tokenizer(adata, dataset_name):
    """Writes a Loom file using loompy directly (anndata.write_loom is deprecated)."""
    adata_tok = adata.copy()
    X = adata_tok.layers['counts'].copy()
    if sp.issparse(X): X = X.toarray()
    X = X.astype(np.float32)
    ensembl_ids = adata_tok.var['ensembl_id'].values.astype(str)
    n_counts = X.sum(axis=1).astype(np.float32)
    matrix = X.T  # genes × cells
    row_attrs = {'ensembl_id': ensembl_ids, 'gene_name': ensembl_ids}
    col_attrs = {
        'CellID'     : np.array(adata_tok.obs_names.tolist()),
        'n_counts'   : n_counts,
        '_cell_type' : np.array(adata_tok.obs['_cell_type'].values.tolist()),
        '_condition' : np.array(adata_tok.obs['_condition'].values.tolist()),
        '_assay_type': np.array(adata_tok.obs['assay_type'].values.tolist()),
        '_patient_id': np.array(adata_tok.obs['patient_id'].values.tolist())
    }
    loom_path = os.path.join(TOKENIZED_DIR, f'{dataset_name}.loom')
    loompy.create(loom_path, matrix, row_attrs, col_attrs)
    print(f'   Saved loom: {loom_path}  ({matrix.shape[1]} cells, {matrix.shape[0]} genes)')
    return loom_path

loom_paths = []
for i, adata in enumerate(adatas):
    name = f'dataset_{i+1}'
    print(f'\n--- Preparing dataset {i+1} for tokenizer ---')
    loom_paths.append(prepare_for_tokenizer(adata, name))

print('\n Running Geneformer tokenizer...')
tk = TranscriptomeTokenizer(custom_attr_name_dict={'_cell_type':'cell_type','_condition':'condition',
                                                   '_assay_type':'assay_type','_patient_id':'patient_id'}, nproc=4)
TOKENIZED_DATA_DIR = os.path.join(OUTPUT_DIR, 'tokenized_data')
os.makedirs(TOKENIZED_DATA_DIR, exist_ok=True)
tk.tokenize_data(data_directory=TOKENIZED_DIR, output_directory=TOKENIZED_DATA_DIR, output_prefix='geneformer_input', file_format='loom')
print('✅ Tokenization complete.')
print(f'   Tokenized dataset saved to: {TOKENIZED_DATA_DIR}')

## 6. Generate Geneformer Cell Embeddings

In [ ]:
import sys
if '/content/Geneformer' not in sys.path:
    sys.path.insert(0, '/content/Geneformer')
import os, torch, pandas as pd
from geneformer import EmbExtractor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
EMB_DIR = os.path.join(OUTPUT_DIR, 'embeddings')
os.makedirs(EMB_DIR, exist_ok=True)
DATASET_PATH = os.path.join(TOKENIZED_DATA_DIR, 'geneformer_input.dataset')

# ── Option A: Load pre-computed embeddings from disk ───────────────────
# Set SAVED_EMBEDDINGS_PATH in the Config cell to skip re-extraction.
def load_embeddings_from_file(path):
    """Load embeddings saved as .parquet or .csv.
    Required columns: cell_type, condition  +  numeric emb_* columns.
    """
    print(f'   Loading embeddings from: {path}')
    if path.endswith('.parquet'):
        df = pd.read_parquet(path)
    elif path.endswith('.csv'):
        df = pd.read_csv(path, index_col=0)
    else:
        raise ValueError(f'Unsupported format: {path}  (expected .parquet or .csv)')
    missing = [c for c in ['cell_type', 'condition'] if c not in df.columns]
    if missing:
        raise ValueError(f'Saved embedding file missing columns: {missing}')
    print(f'   Shape: {df.shape}  |  '
          f'Conditions: {df["condition"].unique().tolist()}  |  '
          f'Cell types: {df["cell_type"].nunique()}')
    return df

if SAVED_EMBEDDINGS_PATH and os.path.exists(SAVED_EMBEDDINGS_PATH):
    # ── Load from saved file ─────────────────────────────────────────────
    print('📂 Loading pre-computed embeddings (skipping EmbExtractor)...')
    embeddings = load_embeddings_from_file(SAVED_EMBEDDINGS_PATH)
    print('✅ Embeddings loaded from saved file.')
else:
    if SAVED_EMBEDDINGS_PATH:
        print(f'⚠️  SAVED_EMBEDDINGS_PATH set but file not found: {SAVED_EMBEDDINGS_PATH}')
        print('   Falling back to EmbExtractor...')
    # ── Option B: Extract embeddings with Geneformer EmbExtractor ──────
    print(' Extracting embeddings with Geneformer EmbExtractor...')
    embex = EmbExtractor(
        model_type='Pretrained', num_classes=0, emb_mode='cell',
        cell_emb_style='mean_pool',
        filter_data={'cell_type': FOCUS_CELLTYPES, 'assay_type': 'snRNA-seq'},
        max_ncells=None, emb_layer=-1,
        emb_label=['cell_type', 'condition','patient_id'], nproc=4,
    )
    embeddings = embex.extract_embs(
        model_directory='ctheodoris/Geneformer',
        input_data_file=DATASET_PATH,
        output_directory=EMB_DIR,
        output_prefix='geneformer_embs',
    )
    # Save for future reuse
    emb_save_path = os.path.join(EMB_DIR, 'geneformer_embs.parquet')
    embeddings.to_parquet(emb_save_path, index=True)
    print(f'   Embeddings saved to: {emb_save_path}')
    print(f'   ➡️  Set SAVED_EMBEDDINGS_PATH = "{emb_save_path}" in Config to reuse.')
    print('✅ Embeddings extracted.')

print(f'   Embedding matrix shape: {embeddings.shape}')
print(f'   Columns sample: {embeddings.columns.tolist()[:5]}')


## 7. Separation Test (Liu et al.)

In [ ]:
embeddings['cell_type'] = embeddings['cell_type'].str.replace('/', '_', regex=False)
if FOCUS_CELLTYPES:
    FOCUS_CELLTYPES = [ct.replace('/', '_') for ct in FOCUS_CELLTYPES]
#cell_types_to_test = [ct.replace('/', '_') for ct in cell_types_to_test]
import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 0.8,
})
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')


def run_separation_test_publishable(
        embeddings_df, cell_type, control_label, target_label,
        emb_cols=None, donor_col='donor'):
    """
    Separation test with patient-aware statistics for publication.

    Computes:
    - Per-cell cosine similarity margin (Liu et al. Step 6)
    - Mann-Whitney U test (two-tailed composite p-value)
    - Patient-level Cohen's d (guards against pseudoreplication)
    - Balanced accuracy from cross-validated logistic classifier

    Returns dict with: passes, p_value, sep_score, cohens_d,
    balanced_accuracy, n_ctrl, n_tgt, n_donors_ctrl, n_donors_tgt,
    ctrl_margins, tgt_margins.
    """
    ct_mask   = embeddings_df['cell_type'] == cell_type
    ctrl_mask = ct_mask & (embeddings_df['condition'] == control_label)
    tgt_mask  = ct_mask & (embeddings_df['condition'] == target_label)
    n_ctrl, n_tgt = ctrl_mask.sum(), tgt_mask.sum()

    empty = {'passes': False, 'p_value': 1.0, 'sep_score': 0.0,
             'cohens_d': None, 'balanced_accuracy': 0.5,
             'n_ctrl': n_ctrl, 'n_tgt': n_tgt,
             'n_donors_ctrl': 0, 'n_donors_tgt': 0,
             'ctrl_margins': np.array([]), 'tgt_margins': np.array([])}

    if n_ctrl < MIN_CELLS_PER_STATE or n_tgt < MIN_CELLS_PER_STATE:
        print(f'   {cell_type} [{control_label} vs {target_label}]: '
              f'insufficient cells (ctrl={n_ctrl}, tgt={n_tgt}). Skipping.')
        return empty

    if emb_cols is None:
        meta = {'cell_type', 'condition', 'donor', 'cell_id', 'index'}
        emb_cols = [c for c in embeddings_df.columns
                    if c not in meta
                    and pd.api.types.is_numeric_dtype(embeddings_df[c])]

    ctrl_embs = embeddings_df.loc[ctrl_mask, emb_cols].values.astype(np.float32)
    tgt_embs  = embeddings_df.loc[tgt_mask,  emb_cols].values.astype(np.float32)
    ctrl_centroid = ctrl_embs.mean(axis=0, keepdims=True)
    tgt_centroid  = tgt_embs.mean(axis=0,  keepdims=True)

    ctrl_margins = (cosine_similarity(ctrl_embs, ctrl_centroid).flatten() -
                    cosine_similarity(ctrl_embs, tgt_centroid).flatten())
    tgt_margins  = (cosine_similarity(tgt_embs,  tgt_centroid).flatten() -
                    cosine_similarity(tgt_embs,  ctrl_centroid).flatten())

    _, p_ctrl = mannwhitneyu(ctrl_margins, np.zeros(len(ctrl_margins)),
                              alternative='greater')
    _, p_tgt  = mannwhitneyu(tgt_margins,  np.zeros(len(tgt_margins)),
                              alternative='greater')
    sep_score  = (ctrl_margins.mean() + tgt_margins.mean()) / 2
    p_combined = max(p_ctrl, p_tgt)

    # Patient-level Cohen's d
    n_donors_ctrl = n_donors_tgt = 0
    cohens_d = None
    actual_donor_col = None
    for dc in ([donor_col] if donor_col else []) + ['donor', 'Donor', 'patient_id']:
        if dc and dc in embeddings_df.columns:
            actual_donor_col = dc
            break

    if actual_donor_col:
        ctrl_donors = embeddings_df.loc[ctrl_mask, actual_donor_col].values
        tgt_donors  = embeddings_df.loc[tgt_mask,  actual_donor_col].values
        n_donors_ctrl = len(np.unique(ctrl_donors))
        n_donors_tgt  = len(np.unique(tgt_donors))
        ctrl_donor_means = (
            pd.Series(ctrl_margins, index=ctrl_donors).groupby(level=0).mean()
        )
        tgt_donor_means  = (
            pd.Series(tgt_margins,  index=tgt_donors).groupby(level=0).mean()
        )
        if len(ctrl_donor_means) >= 2 and len(tgt_donor_means) >= 2:
            pooled_sd = np.sqrt(
                (ctrl_donor_means.std()**2 + tgt_donor_means.std()**2) / 2 + 1e-10
            )
            cohens_d = float(
                (tgt_donor_means.mean() - ctrl_donor_means.mean()) / pooled_sd
            )

    # Balanced accuracy (linear classifier, 3-fold CV)
    X_all = np.vstack([ctrl_embs, tgt_embs])
    y_all = np.array([0]*len(ctrl_embs) + [1]*len(tgt_embs))
    ba_scores = []
    n_splits  = min(3, min(n_ctrl, n_tgt) // 10)
    if n_splits >= 2:
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        for train_idx, test_idx in skf.split(X_all, y_all):
            clf = LogisticRegression(max_iter=500, C=0.1, random_state=42,
                                      solver='lbfgs')
            clf.fit(X_all[train_idx], y_all[train_idx])
            ba_scores.append(balanced_accuracy_score(
                y_all[test_idx], clf.predict(X_all[test_idx])
            ))
    balanced_accuracy = float(np.mean(ba_scores)) if ba_scores else 0.5

    passes = (p_combined < 0.05) and (sep_score > 0)

    return {
        'passes':             passes,
        'p_value':            float(p_combined),
        'sep_score':          float(sep_score),
        'cohens_d':           cohens_d,
        'balanced_accuracy':  balanced_accuracy,
        'n_ctrl':             int(n_ctrl),
        'n_tgt':              int(n_tgt),
        'n_donors_ctrl':      int(n_donors_ctrl),
        'n_donors_tgt':       int(n_donors_tgt),
        'ctrl_margins':       ctrl_margins,
        'tgt_margins':        tgt_margins,
    }


# ── Setup ──────────────────────────────────────────────────────────────────────
emb_meta_cols    = {'cell_type', 'condition', 'cell_id', 'donor'}
emb_feature_cols = [c for c in embeddings.columns
                    if c not in emb_meta_cols
                    and pd.api.types.is_numeric_dtype(embeddings[c])]

cell_types_to_test = (FOCUS_CELLTYPES if FOCUS_CELLTYPES
                      else embeddings['cell_type'].unique().tolist())
cell_types_to_test = [ct for ct in cell_types_to_test
                       if ct in embeddings['cell_type'].values]

_donor_col_in_emb = None
for _dc in ['donor', 'Donor', 'donor_id', 'patient_id', 'sample_id']:
    if _dc in embeddings.columns:
        _donor_col_in_emb = _dc
        print(f'   Using donor column in embeddings: "{_donor_col_in_emb}"')
        break

# ── States to test ─────────────────────────────────────────────────────────────
# Always test control vs goal.
# With alt states also test control vs alt and goal vs alt.
_alt_states = ALT_STATES if "ALT_STATES" in dir() and ALT_STATES else []

state_pairs = [(CONTROL_LABEL, TARGET_LABEL,
                f"{CONTROL_LABEL} vs {TARGET_LABEL}")]
for alt_state in _alt_states:
    if alt_state in embeddings['condition'].unique():
        state_pairs.append(
            (CONTROL_LABEL, alt_state,
             f"{CONTROL_LABEL} vs {alt_state}")
        )
        state_pairs.append(
            (TARGET_LABEL, alt_state,
             f"{TARGET_LABEL} vs {alt_state}")
        )
    else:
        print(f"   Alt state '{alt_state}' not found in "
              f"embeddings['condition'] — skipping.")

print(f"\nState comparisons to test: {[p[2] for p in state_pairs]}")
print(f"Cell types to test: {cell_types_to_test}")

# ── Run all separation tests ───────────────────────────────────────────────────
separation_results = {}
all_sep_rows       = []

print(f'\n Running separation tests...')

for ct in cell_types_to_test:
    separation_results[ct] = {}
    for label_a, label_b, comp_name in state_pairs:

        has_a = ((embeddings['cell_type'] == ct) &
                 (embeddings['condition'] == label_a)).sum() >= MIN_CELLS_PER_STATE
        has_b = ((embeddings['cell_type'] == ct) &
                 (embeddings['condition'] == label_b)).sum() >= MIN_CELLS_PER_STATE

        if not has_a or not has_b:
            print(f"   {ct} [{comp_name}]: missing condition data — skipping.")
            separation_results[ct][comp_name] = {
                'passes': False, 'p_value': 1.0, 'sep_score': 0.0,
                'cohens_d': None, 'balanced_accuracy': 0.5,
                'n_ctrl': 0, 'n_tgt': 0,
                'n_donors_ctrl': 0, 'n_donors_tgt': 0,
                'ctrl_margins': np.array([]), 'tgt_margins': np.array([])
            }
            continue

        res = run_separation_test_publishable(
            embeddings, ct, label_a, label_b,
            emb_cols=emb_feature_cols,
            donor_col=_donor_col_in_emb
        )
        separation_results[ct][comp_name] = res

        all_sep_rows.append({
            'cell_type':         ct,
            'comparison':        comp_name,
            'label_a':           label_a,
            'label_b':           label_b,
            'passes':            res['passes'],
            'sep_score':         round(res['sep_score'], 5),
            'p_value':           res['p_value'],
            'cohens_d':          round(res['cohens_d'], 3)
                                 if res['cohens_d'] is not None else None,
            'balanced_accuracy': round(res['balanced_accuracy'], 3),
            'n_a':               res['n_ctrl'],
            'n_b':               res['n_tgt'],
            'n_donors_a':        res['n_donors_ctrl'],
            'n_donors_b':        res['n_donors_tgt'],
        })

        d_str  = (f"{res['cohens_d']:.3f}"
                  if res['cohens_d'] is not None else 'N/A')
        status = '✅ PASS' if res['passes'] else '❌ FAIL'
        print(f"   {ct} [{comp_name}]: {status}  "
              f"sep={res['sep_score']:.4f}  "
              f"p={res['p_value']:.2e}  "
              f"d={d_str}  "
              f"bal_acc={res['balanced_accuracy']:.3f}  "
              f"n=({res['n_ctrl']},{res['n_tgt']})")

# ── Determine passing cell types ───────────────────────────────────────────────
# A cell type proceeds to ISP only if it passes ALL separation tests
# for all comparisons that have data. This ensures:
#   1. Control vs goal are meaningfully separated (required for primary ISP)
#   2. Control vs alt are meaningfully separated (required for alt state ISP)
#   3. Goal vs alt are meaningfully separated (confirms states are biologically
#      distinct from each other, not just from control)
# Comparisons with missing data (one condition absent) are skipped.

def all_comparisons_pass(ct_results):
    """
    Returns True only if ALL of the following are true:
    - Every comparison in state_pairs has data for this cell type
    - Every comparison passes the separation test
    If any comparison has missing data OR fails, returns False.
    """
    for label_a, label_b, comp_name in state_pairs:
        res = ct_results.get(comp_name, {})
        # Missing data — one or both conditions absent
        if res.get('n_ctrl', 0) == 0 or res.get('n_tgt', 0) == 0:
            return False
        # Fails the separation test
        if not res.get('passes', False):
            return False
    return True

passing_celltypes = [
    ct for ct in cell_types_to_test
    if all_comparisons_pass(separation_results[ct])
]

# Track cell types that pass primary only — useful for diagnostics
primary_comp = f"{CONTROL_LABEL} vs {TARGET_LABEL}"
passing_primary_only = [
    ct for ct in cell_types_to_test
    if separation_results[ct].get(primary_comp, {}).get('passes', False)
    and ct not in passing_celltypes
]

print(f'\n✅ {len(passing_celltypes)}/{len(cell_types_to_test)} cell types pass '
      f'ALL separation tests.')
print(f'   Proceeding with: {passing_celltypes}')
if passing_primary_only:
    print(f'\n   Cell types passing primary only (excluded from ISP):')
    for ct in passing_primary_only:
        failed_comps = [
            comp for comp, res in separation_results[ct].items()
            if res['n_ctrl'] > 0 and res['n_tgt'] > 0
            and not res['passes']
        ]
        print(f'     {ct}: failed — {failed_comps}')

# ── Summary table ──────────────────────────────────────────────────────────────
sep_summary = pd.DataFrame(all_sep_rows)
sep_summary_path = os.path.join(OUTPUT_DIR, 'separation_test_summary.csv')
sep_summary.to_csv(sep_summary_path, index=False)

print('\nSeparation Test Summary:')
display(sep_summary[['cell_type', 'comparison', 'passes', 'sep_score',
                      'p_value', 'cohens_d', 'balanced_accuracy',
                      'n_a', 'n_b']].sort_values(
    ['cell_type', 'comparison']
).to_string(index=False))

# ── Backward compatibility ─────────────────────────────────────────────────────
# Remap primary comparison result to top-level keys so downstream code
# that accesses separation_results[ct]['passes'] directly still works.
for ct in cell_types_to_test:
    primary_res = separation_results[ct].get(primary_comp, {})
    for key, val in primary_res.items():
        if key not in separation_results[ct]:
            separation_results[ct][key] = val

In [ ]:
# ── Separation Test Figures ──────────────────────────────────────
# With alt states this produces:
#   Figure 1: Per-cell-type panel for each comparison (PCA + violin)
#   Figure 2: Multi-cell-type summary bar charts per comparison
#   Figure 3: Alt state comparison matrix (if alt states defined)

SEP_FIG_DIR = os.path.join(OUTPUT_DIR, 'separation_test_figures')
os.makedirs(SEP_FIG_DIR, exist_ok=True)

SIG_GREEN = '#2ecc71'
FAIL_RED  = '#e74c3c'

# Build colour palette covering all states
_alt_states = ALT_STATES if "ALT_STATES" in dir() and ALT_STATES else []
_all_states = [CONTROL_LABEL, TARGET_LABEL] + list(_alt_states)
_base_colors = ['#4C72B0', '#DD8452', '#59A14F', '#B07AA1',
                '#E15759', '#76B7B2', '#F28E2B']
PALETTE = {state: _base_colors[i % len(_base_colors)]
           for i, state in enumerate(_all_states)}

# All comparison pairs that were actually tested
_tested_pairs = list(state_pairs)   # state_pairs defined in sep test cell

# ── Compute 2D PCA coordinates (once, shared across all figures) ──────────────
pca = PCA(n_components=2, random_state=42)
all_emb_vals = embeddings[emb_feature_cols].values
coords_2d    = pca.fit_transform(all_emb_vals)
emb_plot = embeddings.copy()
emb_plot['PC1'] = coords_2d[:, 0]
emb_plot['PC2'] = coords_2d[:, 1]

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 1: Per-cell-type panels — one figure per comparison
# Each figure has n_celltypes rows × 2 columns (PCA scatter | cosine violin)
# ══════════════════════════════════════════════════════════════════════════════
print("Generating Figure 1: Per-cell-type separation panels...")

for label_a, label_b, comp_name in _tested_pairs:
    cts_with_data = [
        ct for ct in cell_types_to_test
        if comp_name in separation_results.get(ct, {})
        and separation_results[ct][comp_name]['n_ctrl'] > 0
        and separation_results[ct][comp_name]['n_tgt'] > 0
    ]
    if not cts_with_data:
        print(f"   No cell types with data for [{comp_name}] — skipping.")
        continue

    n_cts = len(cts_with_data)
    fig, axes = plt.subplots(n_cts, 2,
                              figsize=(13, 4.5 * n_cts),
                              squeeze=False)
    fig.suptitle(
        f'Geneformer Cell-State Separation Test\n{comp_name}',
        fontsize=14, fontweight='bold', y=1.01
    )

    pair_palette = {label_a: PALETTE[label_a], label_b: PALETTE[label_b]}

    for row_idx, ct in enumerate(cts_with_data):
        res     = separation_results[ct][comp_name]
        ct_mask = (emb_plot['cell_type'] == ct) & \
                  (emb_plot['condition'].isin([label_a, label_b]))
        ct_df   = emb_plot[ct_mask]

        # Panel A: PCA scatter
        ax = axes[row_idx, 0]
        for cond, color in pair_palette.items():
            m = ct_df['condition'] == cond
            ax.scatter(ct_df.loc[m, 'PC1'], ct_df.loc[m, 'PC2'],
                       c=color, alpha=0.45, s=14, linewidths=0,
                       label=cond, rasterized=True)
        ax.set_xlabel(
            f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=10)
        ax.set_ylabel(
            f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=10)

        # Title colour: green = passes all comparisons, red = fails any
        passes_all  = ct in passing_celltypes
        title_color = SIG_GREEN if passes_all else FAIL_RED
        ax.set_title(f'{ct}\nPCA Embedding Space',
                     fontsize=10, fontweight='bold', color=title_color)
        ax.legend(fontsize=9, markerscale=1.5, framealpha=0.8)

        d_str      = (f"Cohen's d = {res['cohens_d']:.2f}"
                      if res['cohens_d'] is not None else "Cohen's d = N/A")
        status_str = 'PASS ✅' if res['passes'] else 'FAIL ❌'
        ax.text(0.04, 0.97,
                f"Sep. score = {res['sep_score']:.4f}\n"
                f"p = {res['p_value']:.2e}\n"
                f"{d_str}\n"
                f"Bal. acc = {res['balanced_accuracy']:.2f}\n"
                f"[{status_str}]",
                transform=ax.transAxes, fontsize=8, va='top',
                bbox=dict(boxstyle='round,pad=0.4', fc='white',
                          alpha=0.85, ec='#cccccc'))

        # Panel B: Cosine margin violins
        ax2 = axes[row_idx, 1]
        margin_parts = []
        if len(res['ctrl_margins']) > 0:
            margin_parts.append(pd.DataFrame({
                'Cosine Margin': res['ctrl_margins'],
                'Condition': label_a
            }))
        if len(res['tgt_margins']) > 0:
            margin_parts.append(pd.DataFrame({
                'Cosine Margin': res['tgt_margins'],
                'Condition': label_b
            }))
        if margin_parts:
            margin_df = pd.concat(margin_parts, ignore_index=True)
            sns.violinplot(data=margin_df, x='Condition', y='Cosine Margin',
                           palette=pair_palette, ax=ax2, inner='box',
                           linewidth=1.2, cut=0, saturation=0.85)
            ax2.axhline(0, color='black', lw=1.2, ls='--', alpha=0.6,
                        label='Separation boundary')
            ax2.set_xlabel('', fontsize=10)
            ax2.set_ylabel(
                'Cosine Similarity Margin\n(toward own centroid)', fontsize=9)
            ax2.set_title(f'{ct}\nCell-State Cosine Margins',
                          fontsize=10, fontweight='bold')
            ax2.legend(fontsize=8, loc='lower right')

    plt.tight_layout()
    safe_comp = comp_name.replace(' ', '_').replace('/', '_')
    fig1_path = os.path.join(
        SEP_FIG_DIR, f'separation_per_celltype_{safe_comp}.pdf'
    )
    plt.savefig(fig1_path, dpi=300, bbox_inches='tight', format='pdf')
    plt.savefig(fig1_path.replace('.pdf', '.png'), dpi=200, bbox_inches='tight')
    plt.show()
    print(f'   Saved: {fig1_path}')

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 2: Multi-cell-type summary — one figure per comparison
# Three panels: separation score | balanced accuracy | Cohen's d
# Bar colours: green = passes all comparisons, amber = passes this only,
#              red = fails this comparison
# ══════════════════════════════════════════════════════════════════════════════
print("\nGenerating Figure 2: Multi-cell-type summary bar charts...")

AMBER = '#F5A623'

def bar_color(ct, comp):
    if not separation_results[ct][comp]['passes']:
        return FAIL_RED
    if ct in passing_celltypes:
        return SIG_GREEN
    return AMBER   # passes this comparison but fails another

for label_a, label_b, comp_name in _tested_pairs:
    cts_for_comp = [
        ct for ct in cell_types_to_test
        if comp_name in separation_results.get(ct, {})
        and separation_results[ct][comp_name]['n_ctrl'] > 0
        and separation_results[ct][comp_name]['n_tgt'] > 0
    ]
    if len(cts_for_comp) < 2:
        print(f"   Fewer than 2 cell types for [{comp_name}] — skipping.")
        continue

    ct_names   = cts_for_comp
    sep_scores = [separation_results[ct][comp_name]['sep_score']
                  for ct in ct_names]
    bal_accs   = [separation_results[ct][comp_name]['balanced_accuracy']
                  for ct in ct_names]
    cohens_ds  = [(separation_results[ct][comp_name]['cohens_d'] or 0.0)
                  for ct in ct_names]
    pvals      = [separation_results[ct][comp_name]['p_value']
                  for ct in ct_names]
    colors     = [bar_color(ct, comp_name) for ct in ct_names]

    fig2, axes2 = plt.subplots(
        1, 3, figsize=(16, max(4, len(ct_names)*0.5 + 2))
    )
    fig2.suptitle(
        f'Separation Test Summary: {comp_name}',
        fontsize=13, fontweight='bold'
    )

    # Panel A: Separation scores
    ax = axes2[0]
    bars = ax.barh(ct_names, sep_scores, color=colors,
                   edgecolor='black', linewidth=0.6)
    ax.axvline(0, color='black', lw=1)
    ax.set_xlabel('Separation Score', fontsize=11)
    ax.set_title('A. Cosine Separation Score',
                 fontsize=11, fontweight='bold')
    for bar, pv in zip(bars, pvals):
        sig = ('***' if pv < 0.001 else '**' if pv < 0.01
               else '*' if pv < 0.05 else 'ns')
        ax.text(bar.get_width() + max(sep_scores)*0.02,
                bar.get_y() + bar.get_height()/2,
                sig, va='center', fontsize=10)

    # Panel B: Balanced accuracy
    ax2b = axes2[1]
    ax2b.barh(ct_names, bal_accs, color=colors,
              edgecolor='black', linewidth=0.6)
    ax2b.axvline(0.5, color='gray', lw=1.2, ls='--', alpha=0.7,
                 label='Chance (0.5)')
    ax2b.set_xlim(0, 1.05)
    ax2b.set_xlabel('Balanced Accuracy (3-fold CV)', fontsize=11)
    ax2b.set_title('B. Linear Classifier Accuracy',
                   fontsize=11, fontweight='bold')
    ax2b.legend(fontsize=8)

    # Panel C: Cohen's d
    ax3 = axes2[2]
    ax3.barh(ct_names, cohens_ds, color=colors,
             edgecolor='black', linewidth=0.6)
    ax3.axvline(0,   color='black', lw=1)
    ax3.axvline(0.5, color='gray', lw=1, ls=':', alpha=0.6,
                label='d=0.5 (medium)')
    ax3.axvline(0.8, color='gray', lw=1, ls='--', alpha=0.6,
                label='d=0.8 (large)')
    ax3.set_xlabel("Cohen's d (patient-level)", fontsize=11)
    ax3.set_title('C. Patient-Level Effect Size',
                  fontsize=11, fontweight='bold')
    ax3.legend(fontsize=8)

    legend_handles = [
        mpatches.Patch(color=SIG_GREEN,
                       label='Passes all comparisons'),
        mpatches.Patch(color=AMBER,
                       label='Passes this comparison only'),
        mpatches.Patch(color=FAIL_RED,
                       label='Fails this comparison'),
    ]
    fig2.legend(handles=legend_handles, loc='lower center', ncol=3,
                fontsize=9, framealpha=0.9, bbox_to_anchor=(0.5, -0.06))

    plt.tight_layout()
    safe_comp = comp_name.replace(' ', '_').replace('/', '_')
    fig2_path = os.path.join(
        SEP_FIG_DIR, f'separation_summary_{safe_comp}.pdf'
    )
    plt.savefig(fig2_path, dpi=300, bbox_inches='tight', format='pdf')
    plt.savefig(fig2_path.replace('.pdf', '.png'), dpi=200, bbox_inches='tight')
    plt.show()
    print(f'   Saved: {fig2_path}')

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 3: Alt state comparison matrix (only if alt states defined)
# Heatmap of separation scores + pass/fail matrix
# across all cell types × all comparisons simultaneously
# ══════════════════════════════════════════════════════════════════════════════
if _alt_states and len(_tested_pairs) > 1:
    print("\nGenerating Figure 3: Alt state comparison matrix...")

    comp_names  = [p[2] for p in _tested_pairs]
    matrix_data = pd.DataFrame(index=cell_types_to_test, columns=comp_names,
                                dtype=float)
    pass_matrix = pd.DataFrame(index=cell_types_to_test, columns=comp_names,
                                dtype=bool)

    for ct in cell_types_to_test:
        for comp_name in comp_names:
            if comp_name in separation_results.get(ct, {}):
                res = separation_results[ct][comp_name]
                if res['n_ctrl'] > 0 and res['n_tgt'] > 0:
                    matrix_data.loc[ct, comp_name] = res['sep_score']
                    pass_matrix.loc[ct, comp_name] = res['passes']
                else:
                    matrix_data.loc[ct, comp_name] = np.nan
                    pass_matrix.loc[ct, comp_name] = False
            else:
                matrix_data.loc[ct, comp_name] = np.nan
                pass_matrix.loc[ct, comp_name] = False

    fig3, axes3 = plt.subplots(1, 2, figsize=(
        max(10, len(comp_names)*3 + 2),
        max(5, len(cell_types_to_test)*0.7 + 2)
    ))
    fig3.suptitle('Separation Test Matrix — All States × All Cell Types',
                  fontsize=13, fontweight='bold')

    # Panel A: Separation score heatmap
    ax_heat = axes3[0]
    mat_vals = matrix_data.astype(float)
    vmax = max(0.01, float(mat_vals.max().max()))
    im = ax_heat.imshow(mat_vals.values, aspect='auto',
                        cmap='RdYlGn', vmin=0, vmax=vmax)
    ax_heat.set_xticks(range(len(comp_names)))
    ax_heat.set_xticklabels(comp_names, rotation=35, ha='right', fontsize=8)
    ax_heat.set_yticks(range(len(cell_types_to_test)))
    ax_heat.set_yticklabels(cell_types_to_test, fontsize=9)
    ax_heat.set_title('A. Separation Score Heatmap',
                      fontsize=11, fontweight='bold')
    plt.colorbar(im, ax=ax_heat, label='Sep. Score', shrink=0.8)

    for i, ct in enumerate(cell_types_to_test):
        for j, comp in enumerate(comp_names):
            val = mat_vals.loc[ct, comp]
            if not np.isnan(val):
                marker = '✓' if pass_matrix.loc[ct, comp] else '✗'
                ax_heat.text(j, i, f'{val:.4f}\n{marker}',
                             ha='center', va='center', fontsize=6.5,
                             color='black' if val < vmax*0.6 else 'white')

    # Panel B: Pass/fail matrix
    ax_pass = axes3[1]
    pass_vals = pass_matrix.astype(float).values
    im2 = ax_pass.imshow(pass_vals, aspect='auto',
                         cmap='RdYlGn', vmin=0, vmax=1)
    ax_pass.set_xticks(range(len(comp_names)))
    ax_pass.set_xticklabels(comp_names, rotation=35, ha='right', fontsize=8)
    ax_pass.set_yticks(range(len(cell_types_to_test)))
    ax_pass.set_yticklabels(cell_types_to_test, fontsize=9)
    ax_pass.set_title('B. Pass / Fail Matrix\n'
                      '(green = passes, red = fails)',
                      fontsize=11, fontweight='bold')

    for i, ct in enumerate(cell_types_to_test):
        for j, comp in enumerate(comp_names):
            passes = pass_matrix.loc[ct, comp]
            ax_pass.text(j, i, '✓' if passes else '✗',
                         ha='center', va='center',
                         fontsize=12, fontweight='bold',
                         color='white')

    # Highlight rows for cell types that pass ALL comparisons
    for i, ct in enumerate(cell_types_to_test):
        if ct in passing_celltypes:
            for ax_tmp in [ax_heat, ax_pass]:
                ax_tmp.add_patch(plt.Rectangle(
                    (-0.5, i-0.5), len(comp_names), 1,
                    fill=False, edgecolor=SIG_GREEN, lw=2.5
                ))

    plt.tight_layout()
    fig3_path = os.path.join(SEP_FIG_DIR, 'separation_matrix_all_states.pdf')
    plt.savefig(fig3_path, dpi=300, bbox_inches='tight', format='pdf')
    plt.savefig(fig3_path.replace('.pdf', '.png'), dpi=200, bbox_inches='tight')
    plt.show()
    print(f'   Saved: {fig3_path}')

print('\n✅ Separation test figures saved.')

## 8. Select Candidate Perturbation Genes

In [ ]:
# Deduplicate while preserving order
MASH_GENES = [

    # ── GWAS / genetic risk (10) ──────────────────────────────────────────────
    # Validated GWAS loci — expression is context-dependent, not constitutive.
    # Independent of expression datasets — strongest unbiased evidence.
    "PNPLA3",    # rs738409 I148M — largest MASH risk locus; lipid droplet
    "TM6SF2",    # E167K — VLDL secretion defect; steatosis + fibrosis
    "MBOAT7",    # rs641738 — phosphatidylinositol remodelling
    "HSD17B13",  # rs72613567 — loss-of-function PROTECTIVE against MASH
    "GCKR",      # rs1260326 — glucokinase regulator; de novo lipogenesis
    "SAMM50",    # rs3761472 — mitochondrial membrane; fibrosis GWAS
    "NCAN",      # rs2228603 — hepatic steatosis GWAS
    "ABCB4",     # rs2109505 — biliary phosphatidylcholine; fibrosis risk
    "CIDEB",     # rs1805081 — lipid droplet fusion; protective variant
    "PTPN11",    # rs11066301 — protein tyrosine phosphatase; MASH GWAS

    # ── ECM / fibrosis / HSC activation (12) ─────────────────────────────────
    # Near-absent in healthy liver; specifically upregulated in activated HSCs.
    "COL1A1",    # fibrillar collagen — canonical fibrosis hallmark
    "COL1A2",
    "COL3A1",    # reticular collagen
    "ACTA2",     # α-SMA — activated HSC marker; absent in quiescent HSCs
    "LOXL1",     # collagen crosslinking; M5 fibrosis hub (Piras 2024)
    "LOXL2",
    "TIMP1",     # MMP inhibitor — blocks fibrosis resolution
    "TIMP2",
    "MMP2",      # matrix metalloproteinase — ECM remodelling
    "MMP9",
    "PDGFRB",    # PDGF receptor β — HSC proliferation; upregulated on activation
    "SMOC2",     # M5 fibrosis coexpression module driver (Piras & DiStefano 2024)

    # ── TGF-β / SMAD fibrogenic signalling (5) ───────────────────────────────
    "TGFB1",     # master pro-fibrotic cytokine; regulated not constitutive
    "TGFB2",
    "SMAD2",     # canonical fibrogenic effector
    "SMAD3",
    "TGFBR1",    # ALK5 — TGF-β type I receptor

    # ── Innate immune / Kupffer cells / NLRP3 (12) ───────────────────────────
    # All inducible — not constitutively expressed at high levels.
    "TNF",       # TNFα — inducible; central MASH pro-inflammatory cytokine
    "IL6",       # inducible; elevated in MASH
    "IL1B",      # NLRP3 product; specifically upregulated in MASH
    "IL18",      # NLRP3 product; elevated in MASH
    "CXCL10",    # IP-10 — IFN-induced; markedly elevated in MASH
    "CCL2",      # MCP-1 — monocyte recruitment; induced in MASH
    "CCR2",      # MCP-1 receptor on infiltrating monocytes
    "CD68",      # macrophage marker; regulated by activation state
    "TLR4",      # LPS receptor; key innate immune MASH driver
    "MYD88",     # central TLR adaptor; regulated
    "NLRP3",     # inflammasome sensor — key MASH driver; therapeutic target
    "CASP1",     # inflammasome effector caspase; IL-1β/IL-18 maturation

    # ── Lipid-associated macrophage (LAM) markers (5) ────────────────────────
    # MASH-specific macrophage subpopulation (Guilliams 2022, 2025).
    # Specifically elevated in MASH — not expressed in healthy liver macrophages.
    "TREM2",     # LAM hub; strongly elevated in MASH macrophages
    "GPNMB",     # glycoprotein NMB; LAM marker; fibrosis correlate
    "SPP1",      # osteopontin; LAM/scar macrophage; MASH progression marker
    "FABP4",     # lipid-associated macrophage marker; disease-regulated
    "LGALS3",    # galectin-3; macrophage activation; fibrosis biomarker

    # ── Adaptive immunity — T cells / checkpoints (4) ────────────────────────
    "CD8A",      # cytotoxic T cell marker
    "FOXP3",     # Treg master TF; regulated by immune context
    "PDCD1",     # PD-1 — T cell exhaustion; elevated in chronic MASH
    "IFNG",      # IFN-γ — inducible Th1 cytokine; Kupffer cell priming

    # ── Lipid metabolism — specifically dysregulated in MASH (9) ─────────────
    # Regulated by nutritional/hormonal state — not constitutively maximal.
    "FASN",      # fatty acid synthase; upregulated in MASH hepatocytes
    "ACACA",     # ACC1 — rate-limiting DNL; induced by insulin/SREBP
    "SREBF1",    # SREBP-1c — master lipogenic TF; induced in MASH
    "DGAT2",     # diacylglycerol acyltransferase; TG synthesis; regulated
    "PLIN2",     # lipid droplet protein; regulated by lipid load
    "CD36",      # FA translocase; markedly upregulated in MASH hepatocytes
    "ACSL4",     # long-chain acyl-CoA synthetase; ferroptosis sensitiser
    "CPT1A",     # FAO rate-limiting step; suppressed in MASH
    "PPARA",     # PPARα — FAO master regulator; most suppressed NR in MASH

    # ── Nuclear receptors / hepatocyte TFs (8) ───────────────────────────────
    # Expression regulated by nutritional state and disease — not constitutive.
    "PPARG",     # PPARγ — lipogenesis; HSC quiescence
    "NR1H4",     # FXR — bile acid/lipid homeostasis; therapeutic target
    "HNF4A",     # central MASH network hub; reduced in advanced disease
    "CEBPA",     # hepatocyte TF; suppressed in MASH fibrosis
    "FOXO1",     # gluconeogenesis + insulin signalling integrator
    "NR0B2",     # SHP — FXR-inducible metabolic gatekeeper
    "THRB",      # THR-β — target of resmetirom (FDA approved 2024)
    "KLF6",      # Krüppel-like factor 6 — HSC activation TF

    # ── Oxidative stress / antioxidant / ferroptosis (8) ─────────────────────
    # Stress-responsive — induced by ROS, not constitutively maximal.
    "NFE2L2",    # NRF2 — stress-inducible master antioxidant TF
    "HMOX1",     # heme oxygenase-1; strongly stress-inducible
    "GPX4",      # ferroptosis gatekeeper; regulated by lipid peroxides
    "SLC7A11",   # xCT — cystine import; ferroptosis; regulated
    "NOX4",      # NADPH oxidase 4; ROS production; fibrosis driver
    "SIRT1",     # NAD+-deacetylase; MASH-suppressed; not constitutive
    "TXNIP",     # thioredoxin-interacting protein; oxidative stress sensor
    "SOD2",      # mitochondrial SOD; regulated by oxidative stress

    # ── Cell death — apoptosis / necroptosis / pyroptosis (6) ────────────────
    # Activated/induced — not constitutively high in healthy hepatocytes.
    "TP53",      # p53 — stress sensor; elevated in MASH
    "CASP3",     # effector caspase — apoptosis executor; activated
    "RIPK3",     # necroptosis kinase; elevated in MASH
    "MLKL",      # necroptosis executor
    "GSDMD",     # gasdermin D — pyroptosis pore; specifically induced
    "BCL2L11",   # BIM — pro-apoptotic BH3-only; regulated

    # ── Bile acid / CYP enzymes (4) ───────────────────────────────────────────
    # Regulated by disease state — not constitutively maximal.
    "CYP7A1",    # rate-limiting BA synthesis; reduced in MASH
    "CYP2E1",    # metabolises FFAs → ROS; upregulated in MASH
    "ABCB11",    # BSEP — bile salt export; reduced in MASH
    "FGF19",     # ileal FXR hormone; steatosis modulator; regulated

    # ── ER stress / UPR (4) ──────────────────────────────────────────────────
    # Near-absent in healthy unstressed hepatocytes; induced by lipotoxicity.
    "DDIT3",     # CHOP — pro-apoptotic; strongly stress-induced in MASH
    "ATF3",      # hub gene MASH/ferroptosis networks (Lin 2025)
    "XBP1",      # IRE1α substrate; UPR TF; spliced under ER stress
    "HSPA5",     # GRP78/BiP — ER chaperone; upregulated in MASH

    # ── Autophagy / mitophagy (3) ─────────────────────────────────────────────
    "BECN1",     # beclin-1 — autophagy initiation; regulated
    "SQSTM1",    # p62 — aggregates in MASH hepatocytes; regulated
    "PINK1",     # mitophagy kinase; regulated by mitochondrial damage

    # ── NF-κB / JAK-STAT / MAPK inflammation hubs (4) ────────────────────────
    # Signalling molecules — activity/expression regulated by disease state.
    "NFKB1",     # NF-κB p50 — master inflammatory TF
    "STAT3",     # JAK-STAT; IL-6 effector; regulated
    "MAPK8",     # JNK1 — stress kinase; insulin resistance; MASH driver
    "JAK2",      # JAK-STAT kinase; regulated

    # ── Insulin resistance / glucose (2) ──────────────────────────────────────
    "INSR",      # insulin receptor — reduced signalling in MASH hepatocytes
    "IRS1",      # insulin receptor substrate 1; regulated

    # ── Novel scRNA-seq / spatial MASH drivers (4) ────────────────────────────
    # Identified from recent 2024-2025 MASH multi-omics studies.
    "EGR1",      # early growth response 1 — stress TF; MASH network hub
    "ZFP36",     # ZFP36/TTP — mRNA stability; anti-inflammatory; regulated
    "NAMPT",     # nicotinamide phosphoribosyltransferase; MASH-regulated
    "GADD45B",   # stress-inducible; MASH biomarker (nomogram study 2021)

]
_seen = set()
_deduped = []
for g in MASH_GENES:
    if g not in _seen:
        _deduped.append(g)
        _seen.add(g)
MASH_GENES = _deduped

print(f"✅ {len(MASH_GENES)} unique literature-curated MASH/liver candidate genes loaded.")
print(f"   (Single shared list — same genes tested across ALL cell types)")


In [ ]:
import os, pickle, numpy as np, pandas as pd
import datasets as hf_datasets

# ── Map MASH_GENES symbols → Ensembl IDs via the combined var table ──────────
all_var = pd.concat([a.var for a in adatas])
sym_to_ensembl = dict(zip(all_var.index, all_var["ensembl_id"]))

def get_geneformer_vocabulary():
    vocab_path = "/content/Geneformer/geneformer/gene_median_dictionary.pkl"
    if os.path.exists(vocab_path):
        with open(vocab_path, "rb") as f:
            return set(pickle.load(f).keys())
    print("   ⚠️  Gene median dictionary not found — skipping vocab filter.")
    return None

gf_vocab = get_geneformer_vocabulary()

# ── Build the shared candidate DataFrame ─────────────────────────────────────
# Maps each gene symbol to its Ensembl ID and Geneformer token ID.
# Genes not in the Geneformer vocabulary or not mappable are dropped.
rows = []
missing_ensembl, missing_token = [], []
for symbol in MASH_GENES:
    eid = sym_to_ensembl.get(symbol)
    if eid is None or (isinstance(eid, float) and np.isnan(eid)):
        missing_ensembl.append(symbol)
        continue
    if gf_vocab and eid not in gf_vocab:
        missing_token.append(symbol)
        continue
    token_id = tk.gene_token_dict.get(eid)
    if token_id is None:
        missing_token.append(symbol)
        continue
    rows.append({"gene_symbol": symbol, "ensembl_id": eid, "token_id": token_id,
                 "n_tokenized_cells": 0})

shared_candidate_df = pd.DataFrame(rows)
print(f"\n✅ Shared candidate gene mapping:")
print(f"   Input genes        : {len(MASH_GENES)}")
print(f"   Mapped to Ensembl  : {len(shared_candidate_df)}")
print(f"   No Ensembl ID      : {len(missing_ensembl)}  {missing_ensembl[:5]}")
print(f"   Not in GF vocab    : {len(missing_token)}   {missing_token[:5]}")

# ── Count token presence per cell type (for reporting only) ──────────────────
# This does NOT filter the candidate list — all mapped genes are tested.
DATASET_PATH = os.path.join(TOKENIZED_DATA_DIR, "geneformer_input.dataset")
tok_dataset = hf_datasets.load_from_disk(DATASET_PATH)

import shutil, os

def sanitise_cell_type(example):
    example['cell_type'] = example['cell_type'].replace('/', '_')
    return example

original_path = TOKENIZED_DATA_DIR + '/geneformer_input.dataset'
tmp_path      = TOKENIZED_DATA_DIR + '/geneformer_input_sanitised.dataset'

# Map to a different path
tok_dataset = tok_dataset.map(sanitise_cell_type, num_proc=1)
tok_dataset = tok_dataset.flatten_indices()
tok_dataset.save_to_disk(tmp_path)

# Remove original and rename tmp to original
shutil.rmtree(original_path)
os.rename(tmp_path, original_path)

# Reload from the renamed path
import datasets as hf_datasets
tok_dataset = hf_datasets.load_from_disk(original_path)

print(f'✅ Tokenised dataset sanitised.')
print(f'   Unique cell types: {set(tok_dataset["cell_type"])}')

candidate_token_set = set(shared_candidate_df["token_id"].tolist())
ct_token_counts = {ct: {} for ct in passing_celltypes}

def count_tokens_batch(batch):
    for input_ids, ct in zip(batch["input_ids"], batch["cell_type"]):
        if ct not in ct_token_counts:
            return batch
        for tid in set(input_ids):
            ct_token_counts[ct][tid] = ct_token_counts[ct].get(tid, 0) + 1
    return batch

ctrl_dataset = tok_dataset.filter(
    lambda x: x["condition"] == CONTROL_LABEL and
              x["cell_type"] in set(passing_celltypes) and
              x["assay_type"] == "snRNA-seq",
    num_proc=4, batch_size=1000
)
ctrl_dataset.map(count_tokens_batch, batched=True, batch_size=100, num_proc=1)

# Report coverage per cell type — same candidate_df used for all
print("\nCandidate gene token presence across cell types:")
for ct in passing_celltypes:
    total_cells = (pd.Series(ctrl_dataset["cell_type"]) == ct).sum()
    present = sum(
        1 for row in shared_candidate_df.itertuples()
        if ct_token_counts.get(ct, {}).get(row.token_id, 0) >= 5
    )
    print(f"   {ct:45s}: {present}/{len(shared_candidate_df)} genes "
          f"in ≥5 cells  (total cells={total_cells})")

# ── Build candidate_dfs: same DataFrame for every passing cell type ───────────
candidate_dfs = {ct: shared_candidate_df.copy() for ct in passing_celltypes}

shared_candidate_df.to_csv(os.path.join(OUTPUT_DIR, "candidate_genes_shared.csv"), index=False)
print(f"\n✅ candidate_dfs built: {len(candidate_dfs)} cell types, "
      f"{len(shared_candidate_df)} shared candidate genes each.")
print("   Saved: candidate_genes_shared.csv")


## 9. In-Silico Perturbation with Geneformer

In [ ]:
import torch
import os

# Verify CUDA is available and set device explicitly
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'CUDA device count: {torch.cuda.device_count()}')

# Force all torch operations to GPU
torch.cuda.set_device(0)
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# Verify
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Warm up the GPU — this forces CUDA initialisation before ISP
_ = torch.zeros(1).cuda()
print(f'GPU warmed up: {torch.cuda.get_device_name(0)}')
print(f'GPU memory after warmup: '
      f'{torch.cuda.memory_allocated()/1e9:.2f}GB allocated, '
      f'{torch.cuda.memory_reserved()/1e9:.2f}GB reserved')

In [ ]:
# Run this once BEFORE the ISP cell
import inspect
import geneformer.in_silico_perturber as isp_module

src = inspect.getsource(isp_module)
device_lines = [
    (i, line.strip()) for i, line in enumerate(src.split('\n'))
    if 'device' in line.lower() and
    any(x in line.lower() for x in ['cuda', 'cpu', 'torch.device', '.to('])
]
for lineno, line in device_lines[:30]:
    print(f'  {lineno:4d}: {line}')

In [ ]:
import sys
if "/content/Geneformer" not in sys.path:
    sys.path.insert(0, "/content/Geneformer")
import os, hashlib, shutil, tempfile, subprocess, pandas as pd
import geneformer.perturber_utils as pu
import datasets as hf_datasets
from geneformer import InSilicoPerturber
import torch
import numpy as np


# ══════════════════════════════════════════════════════════════════════════════
# GPU SETUP
# ══════════════════════════════════════════════════════════════════════════════

def setup_gpu():
    if not torch.cuda.is_available():
        print("⚠️  CUDA not available — ISP will run on CPU (very slow).")
        return torch.device("cpu")

    device = torch.device("cuda:0")
    torch.cuda.set_device(0)

    # expandable_segments prevents memory fragmentation during forward passes.
    # max_split_size_mb is omitted — it was only needed
    # alongside torch.compile which is disabled below.
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    os.environ["TOKENIZERS_PARALLELISM"]   = "false"

    # Warm up CUDA context
    _ = torch.zeros(1, device=device)
    torch.cuda.synchronize()

    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name}  ({vram_gb:.0f} GB VRAM)")

    try:
        util = subprocess.run(
            ["nvidia-smi",
             "--query-gpu=utilization.gpu,memory.used,power.draw",
             "--format=csv,noheader"],
            capture_output=True, text=True, timeout=5
        ).stdout.strip()
        print(f"   GPU status: {util}")
    except Exception:
        pass

    return device


DEVICE = setup_gpu()


# ══════════════════════════════════════════════════════════════════════════════
# PATCH pu.load_model — force GPU placement
#
# Guard prevents recursive patching if this cell is re-run.
#
# torch.compile is intentionally disabled:
#   - reduce-overhead mode uses CUDA Graphs which pre-allocate ~20 GB of
#     private memory pools, causing OOM with full-length sequences.
#   - default mode also triggers CUDA Graph state corruption after OOM,
#     producing AssertionError in cudagraph_trees.py on subsequent runs.
#   - inference_mode + CUDAPrefetchStream (below) provide sufficient speedup
#     without any CUDA Graph involvement.
# ══════════════════════════════════════════════════════════════════════════════

if not getattr(pu, "_load_model_gpu_patched", False):
    # Capture the real original function object directly — NOT via pu.load_model
    # attribute access. After pu.load_model is reassigned, _REAL_LOAD_MODEL
    # still points at the original, preventing infinite recursion.
    _REAL_LOAD_MODEL = pu.load_model

    def _gpu_load_model(model_type, num_classes, model_directory,
                         *args, **kwargs):
        """
        Wraps pu.load_model to guarantee GPU placement after loading.
        Calls _REAL_LOAD_MODEL directly — never pu.load_model — to prevent
        the recursion error that occurs when the patched function calls
        itself via the pu.load_model attribute.
        """
        model = _REAL_LOAD_MODEL(
            model_type, num_classes, model_directory, *args, **kwargs
        )
        if torch.cuda.is_available():
            model = model.to("cuda:0")
            model = model.eval()   # disables dropout; speeds up inference
            print(f"   Model device: {next(model.parameters()).device}")
            # torch.compile intentionally skipped — see note above
            alloc_gb  = torch.cuda.memory_allocated() / 1e9
            reserv_gb = torch.cuda.memory_reserved()  / 1e9
            print(f"   VRAM after model load: "
                  f"{alloc_gb:.2f} GB allocated, "
                  f"{reserv_gb:.2f} GB reserved")
        return model

    pu.load_model = _gpu_load_model
    pu._load_model_gpu_patched = True
    print("✅ pu.load_model patched for GPU (once per session).")
else:
    print("✅ pu.load_model already patched — skipping re-patch.")


# ══════════════════════════════════════════════════════════════════════════════
# CUDA PREFETCH STREAM — overlaps CPU data preparation with GPU inference
# ══════════════════════════════════════════════════════════════════════════════

class CUDAPrefetchStream:
    def __init__(self, iterable, device="cuda:0"):
        self.iterable = iterable
        self.device   = device
        self.stream   = torch.cuda.Stream(device=device)
        self._next    = None

    def _to_device(self, obj):
        if isinstance(obj, torch.Tensor):
            return obj.to(self.device, non_blocking=True)
        if isinstance(obj, dict):
            return {k: self._to_device(v) for k, v in obj.items()}
        if isinstance(obj, (list, tuple)):
            return type(obj)(self._to_device(v) for v in obj)
        return obj

    def _preload(self):
        try:
            item = next(self._iter)
        except StopIteration:
            self._next = None
            return
        with torch.cuda.stream(self.stream):
            self._next = self._to_device(item)

    def __iter__(self):
        self._iter = iter(self.iterable)
        self._preload()
        return self

    def __next__(self):
        torch.cuda.current_stream().wait_stream(self.stream)
        item = self._next
        if item is None:
            raise StopIteration
        self._preload()
        return item

    def __len__(self):
        try:
            return len(self.iterable)
        except TypeError:
            return 0


_OriginalDataLoader = torch.utils.data.DataLoader

if torch.cuda.is_available() and not getattr(
        torch.utils.data.DataLoader, "_prefetch_patched", False):

    class _PrefetchingDataLoader(_OriginalDataLoader):
        _prefetch_patched = True

        def __iter__(self):
            return iter(
                CUDAPrefetchStream(super().__iter__(), device="cuda:0")
            )

    torch.utils.data.DataLoader = _PrefetchingDataLoader
    print("✅ DataLoader patched with CUDA prefetch stream.")
else:
    if not torch.cuda.is_available():
        print("   DataLoader prefetch: skipped (no CUDA).")
    else:
        print("   DataLoader prefetch: already patched.")


# ══════════════════════════════════════════════════════════════════════════════
# PATIENT-AWARE CELL SAMPLING
# ══════════════════════════════════════════════════════════════════════════════

def patient_aware_sample_for_isp(
        dataset, donor_key="donor",
        max_cells_total=500, min_cells_per_donor=5,
        random_state=42):
    """
    Donor-stratified sampling of cell indices from a pre-filtered HuggingFace
    Dataset. Returns (sampled_indices, donor_report).
    """
    rng = np.random.default_rng(random_state)

    if donor_key not in dataset.column_names:
        print(f"   ⚠️  donor_key '{donor_key}' not found — random sampling.")
        n = min(max_cells_total, len(dataset))
        return rng.choice(len(dataset), size=n, replace=False).tolist(), \
               {"all_cells": n}

    donors = dataset[donor_key]
    donor_to_indices = {}
    for i, d in enumerate(donors):
        donor_to_indices.setdefault(d, []).append(i)

    eligible = {d: idxs for d, idxs in donor_to_indices.items()
                if len(idxs) >= min_cells_per_donor}

    if not eligible:
        print(f"   ⚠️  No donors with ≥{min_cells_per_donor} cells "
              f"— random sampling.")
        n = min(max_cells_total, len(dataset))
        return rng.choice(len(dataset), size=n, replace=False).tolist(), \
               {"all_cells": n}

    n_donors = len(eligible)
    cells_per_donor = max(min_cells_per_donor,
                          int(np.floor(max_cells_total / n_donors)))
    sampled, report = [], {}
    for donor, idxs in eligible.items():
        n_take = min(cells_per_donor, len(idxs))
        chosen = rng.choice(idxs, size=n_take, replace=False).tolist()
        sampled.extend(chosen)
        report[donor] = n_take

    if len(sampled) > max_cells_total:
        sampled = rng.choice(sampled, size=max_cells_total,
                             replace=False).tolist()

    print(f"   Patient-aware sampling: {len(sampled)} cells from "
          f"{n_donors} donors (~{cells_per_donor} cells/donor)")
    return sampled, report



def trim_sequences_safe(dataset, max_len=1024, protected_token_ids=None):
    """
    Trim token sequences to max_len while guaranteeing that all candidate
    gene tokens AND Geneformer special tokens are never removed.
    """
    if protected_token_ids is None:
        protected_token_ids = set()

    lengths   = dataset["length"]
    max_orig  = int(np.max(lengths))
    mean_orig = float(np.mean(lengths))

    if max_len >= max_orig:
        print(f"   max_len={max_len} >= max sequence length {max_orig} "
              f"— no trimming needed.")
        return dataset

    # ── Detect Geneformer special tokens from the dataset ────────────────
    # CLS token is always at position 0, EOS token is always at the end.
    # DONT REMOVE or Geneformer's internal loop will crash.
    sample_ids  = dataset["input_ids"][0]
    cls_token   = int(sample_ids[0])   # always first
    eos_token   = int(sample_ids[-1])  # always last

    # Add special tokens to the protected set so the swap logic
    # never overwrites them and the trim never removes them
    fully_protected = protected_token_ids | {cls_token, eos_token}

    print(f"   Special tokens detected — CLS: {cls_token}, EOS: {eos_token}")

    n_trimmed = sum(1 for l in lengths if l > max_len)
    pct       = (1 - max_len / max_orig) * 100
    print(f"   Trimming {n_trimmed}/{len(dataset)} sequences to {max_len} "
          f"tokens ({pct:.0f}% reduction from max {max_orig}). "
          f"Mean: {mean_orig:.0f} → ≤{max_len}.")

    def trim_example(example):
        ids = list(example["input_ids"])
        if len(ids) <= max_len:
            return example

        # Always keep CLS at position 0 and EOS at the last position —
        # extract them, trim the middle, then reattach
        cls = ids[0]
        eos = ids[-1]
        middle = ids[1:-1]   # gene tokens only, no special tokens

        # Target middle length: max_len - 2 (for CLS + EOS)
        target_middle = max_len - 2

        if len(middle) <= target_middle:
            return example

        keep_middle = middle[:target_middle]
        tail_middle = middle[target_middle:]

        # Find candidate tokens that ended up in the tail
        protected_in_tail = [
            tid for tid in tail_middle
            if int(tid) in protected_token_ids   # candidate genes only
        ]

        if protected_in_tail:
            # Swap protected tail tokens into the keep window,
            # replacing the lowest-ranked non-protected positions
            # (working from the end of keep_middle backward)
            swap_positions = [
                i for i in range(len(keep_middle) - 1, -1, -1)
                if int(keep_middle[i]) not in fully_protected
            ]
            for swap_pos, prot_tid in zip(swap_positions, protected_in_tail):
                keep_middle[swap_pos] = prot_tid

        # Reconstruct: CLS + trimmed gene tokens + EOS
        example["input_ids"] = [cls] + keep_middle + [eos]
        example["length"]    = len(example["input_ids"])
        return example

    trimmed = dataset.map(trim_example, num_proc=1)

    # Verify special tokens are intact in a sample
    sample_trimmed = trimmed["input_ids"][0]
    assert int(sample_trimmed[0])  == cls_token, "CLS token missing after trim"
    assert int(sample_trimmed[-1]) == eos_token, "EOS token missing after trim"
    print(f"   ✅ Special tokens intact. "
          f"Sample length after trim: {len(sample_trimmed)}")

    return trimmed
# ══════════════════════════════════════════════════════════════════════════════
# PatchedISP — candidate gene filtering, full sequences, GPU-optimised
# ══════════════════════════════════════════════════════════════════════════════

class PatchedISP(InSilicoPerturber):
    def __init__(self, candidate_token_ids=None, **kwargs):
        kwargs["genes_to_perturb"] = "all"
        self.candidate_token_ids = (
            set(int(t) for t in candidate_token_ids)
            if candidate_token_ids else None
        )
        super().__init__(**kwargs)

    def apply_additional_filters(self, filtered_input_data):
        if self.cell_states_to_model is not None:
            filtered_input_data = pu.filter_data_by_start_state(
                filtered_input_data, self.cell_states_to_model, self.nproc)
        if self.anchor_token is not None:
            filtered_input_data = pu.filter_data_by_tokens_and_log(
                filtered_input_data, self.anchor_token, self.nproc,
                "anchor_gene")
        filtered_input_data = pu.downsample_and_sort(
            filtered_input_data, self.max_ncells)
        if self.cell_inds_to_perturb != "all":
            filtered_input_data = pu.slice_by_inds_to_perturb(
                filtered_input_data, self.cell_inds_to_perturb)
        return filtered_input_data

    def perturb_data(self, model_directory, input_data_file,
                     output_directory, output_prefix):
        # torch.inference_mode disables autograd tracking for the entire run —
        # reduces memory overhead and speeds up each forward pass.
        # No CUDA Graphs are involved so there is no OOM risk.
        with torch.inference_mode():
            if self.candidate_token_ids:
                print("   Filtering to cells containing ≥1 candidate gene "
                      "token (full sequences preserved)...")
                full_dataset  = hf_datasets.load_from_disk(
                    input_data_file, keep_in_memory=True
                )
                candidate_set = self.candidate_token_ids

                def has_candidate(example):
                    return any(int(t) in candidate_set
                               for t in example["input_ids"])

                filtered_dataset = full_dataset.filter(
                    has_candidate, num_proc=1
                )
                n_before = len(full_dataset)
                n_after  = len(filtered_dataset)
                print(f"   {n_after}/{n_before} cells contain "
                      f"≥1 candidate gene token.")

                if n_after == 0:
                    print("   ⚠️  No cells remain. Skipping.")
                    return

                # Compact into a single contiguous Arrow file —
                # faster sequential reads during the perturbation loop
                filtered_dataset = filtered_dataset.flatten_indices()

                tmp_dir  = tempfile.mkdtemp()
                tmp_path = os.path.join(tmp_dir, "candidate_cells.dataset")
                filtered_dataset.save_to_disk(tmp_path)

                try:
                    util = subprocess.run(
                        ["nvidia-smi",
                         "--query-gpu=utilization.gpu,memory.used,power.draw",
                         "--format=csv,noheader"],
                        capture_output=True, text=True, timeout=5
                    ).stdout.strip()
                    # print(f"   GPU before forward passes: {util}")
                    # # Replace this single line:
                    # print(f"   GPU before forward passes: {util}")

                    # With this background monitor that logs every 30s during ISP:
                    # import threading

                    # def _gpu_monitor(stop_event, interval=30):
                    #     while not stop_event.is_set():
                    #         try:
                    #             out = subprocess.run(
                    #                 ["nvidia-smi",
                    #                 "--query-gpu=utilization.gpu,memory.used,power.draw",
                    #                 "--format=csv,noheader"],
                    #                 capture_output=True, text=True, timeout=5
                    #             ).stdout.strip()
                    #             print(f"   [GPU monitor] {out}", flush=True)
                    #         except Exception:
                    #             pass
                    #         stop_event.wait(interval)

                    # _stop = threading.Event()
                    # _monitor_thread = threading.Thread(
                    #     target=_gpu_monitor, args=(_stop,), daemon=True
                    # )
                    # _monitor_thread.start()
                    # print(f"   GPU before forward passes: {util}")

                    # try:
                    #     super().perturb_data(
                    #         model_directory, tmp_path,
                    #         output_directory, output_prefix
                    #     )
                    # finally:
                    #     _stop.set()
                    #     _monitor_thread.join(timeout=5)
                    #     shutil.rmtree(tmp_dir, ignore_errors=True)
                    #     torch.cuda.empty_cache()
                except Exception:
                    pass

                try:
                    super().perturb_data(
                        model_directory, tmp_path,
                        output_directory, output_prefix
                    )
                finally:
                    shutil.rmtree(tmp_dir, ignore_errors=True)
                    torch.cuda.empty_cache()
            else:
                super().perturb_data(
                    model_directory, input_data_file,
                    output_directory, output_prefix
                )
            torch.cuda.empty_cache()


# ══════════════════════════════════════════════════════════════════════════════
# MAIN ISP LOOP
# ══════════════════════════════════════════════════════════════════════════════

ISP_OUTPUT_DIR  = os.path.join(OUTPUT_DIR, "isp_output")
os.makedirs(ISP_OUTPUT_DIR, exist_ok=True)
MODE_MAP        = {"down": "delete", "delete": "delete", "up": "overexpress"}
gf_perturb_type = MODE_MAP.get(PERTURB_MODE, "delete")
DATASET_PATH    = os.path.join(TOKENIZED_DATA_DIR, "geneformer_input.dataset")

print(f"\n⚡ Running Geneformer ISP (GPU-optimised, full token sequences)...")
print(f"   Device            : {DEVICE}")
print(f"   Perturbation mode : {gf_perturb_type}")
print(f"   Shared candidates : {len(shared_candidate_df)} genes")
print(f"   Cell types        : {list(candidate_dfs.keys())}")
print(f"   Control → Target  : {CONTROL_LABEL} → {TARGET_LABEL}")

original_write = pu.write_perturbation_dictionary

def safe_write(data, path):
    basename = os.path.basename(path)
    if len(basename) > 200:
        dir_part    = os.path.dirname(path)
        safe_prefix = basename[:60]
        token_hash  = hashlib.md5(basename.encode()).hexdigest()[:16]
        path        = os.path.join(dir_part, f"{safe_prefix}_{token_hash}")
    original_write(data, path)

pu.write_perturbation_dictionary = safe_write

import time

try:
    for cell_type, candidate_df in candidate_dfs.items():
        t_cell_start = time.time()
        print(f"\n{'─'*60}")
        print(f"Cell type: {cell_type}")

        # ── State embeddings ──────────────────────────────────────────────
        ctrl_mask = ((embeddings["cell_type"] == cell_type) &
                     (embeddings["condition"] == CONTROL_LABEL))
        tgt_mask  = ((embeddings["cell_type"] == cell_type) &
                     (embeddings["condition"] == TARGET_LABEL))

        if ctrl_mask.sum() == 0 or tgt_mask.sum() == 0:
            print(f"   ⚠️  Skipping: insufficient cells for state embeddings.")
            continue

        control_embs = embeddings.loc[ctrl_mask, emb_feature_cols].values
        target_embs  = embeddings.loc[tgt_mask,  emb_feature_cols].values
        alt_state_embs = {}
        for alt_state in ALT_STATES:
            alt_mask = ((embeddings["cell_type"] == cell_type) &
                        (embeddings["condition"] == alt_state))
            if alt_mask.sum() == 0:
                print(f"   ⚠️  No embeddings for alt state '{alt_state}' "
                      f"in {cell_type} — skipping.")
                continue
            alt_embs = embeddings.loc[alt_mask, emb_feature_cols].values
            alt_state_embs[alt_state] = torch.tensor(
                alt_embs.mean(axis=0), dtype=torch.float32
            ).to(DEVICE)

        state_embs_dict_for_isp = {
            CONTROL_LABEL: torch.tensor(
                control_embs.mean(axis=0), dtype=torch.float32
            ).to(DEVICE),
            TARGET_LABEL: torch.tensor(
                target_embs.mean(axis=0), dtype=torch.float32
            ).to(DEVICE),
            **alt_state_embs
        }

        # ── Candidate token IDs ───────────────────────────────────────────
        candidate_token_ids = [
            int(row.token_id) for row in candidate_df.itertuples()
            if hasattr(row, "token_id") and row.token_id is not None
        ]
        if not candidate_token_ids:
            candidate_token_ids = [
                int(tk.gene_token_dict[eid])
                for eid in candidate_df["ensembl_id"]
                if eid in tk.gene_token_dict
            ]
        print(f"   {len(candidate_token_ids)} candidate token IDs.")

        ct_isp_dir = os.path.join(ISP_OUTPUT_DIR, cell_type.replace(" ", "_"))
        os.makedirs(ct_isp_dir, exist_ok=True)

        # ── Step 1: Load and filter to this cell type / condition ─────────
        full_dataset    = hf_datasets.load_from_disk(DATASET_PATH)
        ct_cond_dataset = full_dataset.filter(
            lambda x: (x["cell_type"] == cell_type and
                       x["condition"] == CONTROL_LABEL and
                       x["assay_type"] == "snRNA-seq"),
            num_proc=1
        )
        print(f"   {len(ct_cond_dataset)} control snRNA-seq cells available.")

        # ── Step 2: Detect donor column ───────────────────────────────────
        _donor_key_isp = None
        for _dk in ["donor", "Donor", "donor_id", "patient_id", "sample_id"]:
            if _dk in ct_cond_dataset.column_names:
                _donor_key_isp = _dk
                print(f"   Donor column: \"{_donor_key_isp}\"")
                break

        # ── Step 3: Patient-aware sampling (unchanged) ────────────────────
        if _donor_key_isp and len(ct_cond_dataset) > 0:
            sampled_idx, donor_report = patient_aware_sample_for_isp(
                ct_cond_dataset,
                donor_key=_donor_key_isp,
                max_cells_total=MAX_ISP_CELLS_TOTAL,
                min_cells_per_donor=MIN_CELLS_PER_DONOR_ISP,
            )
            print(f"   Donor report: {donor_report}")
            pd.DataFrame(
                list(donor_report.items()), columns=["donor", "n_cells"]
            ).to_csv(
                os.path.join(
                    ct_isp_dir,
                    f"patient_sampling_{cell_type.replace(' ', '_')}.csv"
                ),
                index=False
            )
        else:
            print(f"   ⚠️  No donor column — random sampling "
                  f"(max {MAX_ISP_CELLS_TOTAL}).")
            rng         = np.random.default_rng(42)
            n_take      = min(MAX_ISP_CELLS_TOTAL, len(ct_cond_dataset))
            sampled_idx = rng.choice(
                len(ct_cond_dataset), size=n_take, replace=False
            ).tolist()
            donor_report = {"random_fallback": n_take}

        isp_input_dataset = ct_cond_dataset.select(sampled_idx)
        print(f"   ISP input: {len(isp_input_dataset)} patient-balanced cells.")

        # ── Step 4: Pre-filter to cells containing candidate genes ────────
        candidate_set_int = set(int(t) for t in candidate_token_ids)

        def _has_candidate(example):
            return any(int(t) in candidate_set_int
                       for t in example["input_ids"])

        isp_input_dataset = isp_input_dataset.filter(
            _has_candidate, num_proc=1
        )
        print(f"   After candidate filter: {len(isp_input_dataset)} cells.")

        if len(isp_input_dataset) == 0:
            print(f"   ⚠️  No cells with candidate genes — "
                  f"skipping {cell_type}.")
            continue

        isp_input_dataset = trim_sequences_safe(
            isp_input_dataset,
            max_len=1024,
            protected_token_ids=set(int(t) for t in candidate_token_ids),
        )

        # ── Step 5: Compact and save to tmp dir ───────────────────────────
        isp_input_dataset = isp_input_dataset.flatten_indices()

        isp_tmp_dir    = tempfile.mkdtemp(prefix="isp_gpu_")
        isp_input_path = os.path.join(isp_tmp_dir, "patient_balanced.dataset")
        isp_input_dataset.save_to_disk(isp_input_path)

        try:
            # ── Step 6: Run ISP ───────────────────────────────────────────
            cell_states_to_model_for_isp = {
                "state_key":   "condition",
                "start_state": CONTROL_LABEL,
                "goal_state":  TARGET_LABEL,
                "alt_states":  ALT_STATES
            }

            isp = PatchedISP(
                candidate_token_ids=candidate_token_ids,
                perturb_type=gf_perturb_type, perturb_rank_shift=None,
                combos=0, anchor_gene=None,
                model_type="Pretrained", num_classes=0,
                emb_mode="cls", cell_emb_style="mean_pool",
                filter_data={"cell_type": [cell_type],
                             "condition": [CONTROL_LABEL],
                             "assay_type": ["snRNA-seq"]},
                cell_states_to_model=cell_states_to_model_for_isp,
                state_embs_dict=state_embs_dict_for_isp,
                max_ncells=None,
                emb_layer=-1,
                forward_batch_size=64,
                nproc=1,
            )

            isp.perturb_data(
                model_directory="ctheodoris/Geneformer",
                input_data_file=isp_input_path,
                output_directory=ct_isp_dir,
                output_prefix=f"isp_{cell_type.replace(' ', '_')}"
            )

            elapsed = (time.time() - t_cell_start) / 60
            print(f"   ✅ ISP complete for {cell_type} ({elapsed:.1f} min).")

            try:
                util = subprocess.run(
                    ["nvidia-smi",
                     "--query-gpu=utilization.gpu,memory.used,power.draw",
                     "--format=csv,noheader"],
                    capture_output=True, text=True, timeout=5
                ).stdout.strip()
                print(f"   GPU after {cell_type}: {util}")
            except Exception:
                pass

        finally:
            shutil.rmtree(isp_tmp_dir, ignore_errors=True)
            torch.cuda.empty_cache()

finally:
    pu.write_perturbation_dictionary = original_write
    if torch.cuda.is_available():
        torch.utils.data.DataLoader = _OriginalDataLoader
        print("   DataLoader restored.")

print("\n✅ All ISP runs complete.")

## 10. Compute Cosine Shift & Statistical Testing

In [ ]:
import sys
if "/content/Geneformer" not in sys.path:
    sys.path.insert(0, "/content/Geneformer")
import os, pickle, pandas as pd, numpy as np, torch
from collections import defaultdict
from scipy import stats
from statsmodels.stats.multitest import multipletests

# ── Liu et al. ISP statistical method ────────────────────────────────────────
# When cell_states_to_model is set, Geneformer stores pre-computed shifts
# for every state in cell_states_to_model simultaneously in one pickle.
# The pickle structure is: {state_label: {(token_id, emb_key): [shifts]}}
# This cell processes goal state AND all alternate states from the same files.
# ─────────────────────────────────────────────────────────────────────────────

STATS_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "isp_stats")
os.makedirs(STATS_OUTPUT_DIR, exist_ok=True)
all_results = []

token_to_ensembl = {v: k for k, v in tk.gene_token_dict.items()}

MIN_CELLS_PER_GENE    = 20
EFFECT_SIZE_THRESHOLD = 2.5e-4

rng = np.random.default_rng(42)

# ── States to process ─────────────────────────────────────────────────────────
# ALT_STATES is defined in config cell, e.g. ALT_STATES = ["Cirrhosis"]
# If not defined or empty, only TARGET_LABEL is processed (backward compatible)
_alt_states   = ALT_STATES if "ALT_STATES" in dir() and ALT_STATES else []
states_to_run = [TARGET_LABEL] + list(_alt_states)
print(f"States to score: {states_to_run}")


def score_one_state(cell_type, state_label, candidate_df,
                    candidate_token_ids_set, merged_by_state,
                    embeddings, emb_feature_cols):
    """
    Run Liu et al. stats for one cell type / state combination.
    Shared helper called for both goal state and each alt state.
    Returns a scored DataFrame or None if data is unavailable.
    """
    if state_label not in merged_by_state:
        print(f"   [{state_label}] Not in pickles — skipping.")
        return None

    state_dict = merged_by_state[state_label]

    # ── Compute baseline cosine similarity ────────────────────────────────
    ctrl_mask  = ((embeddings["cell_type"] == cell_type) &
                  (embeddings["condition"] == CONTROL_LABEL))
    state_mask = ((embeddings["cell_type"] == cell_type) &
                  (embeddings["condition"] == state_label))

    ctrl_mean = embeddings.loc[ctrl_mask, emb_feature_cols].values.mean(axis=0)

    if state_mask.sum() > 0:
        state_mean = embeddings.loc[state_mask, emb_feature_cols].values.mean(axis=0)
    else:
        print(f"   [{state_label}] No embeddings found — using ctrl centroid.")
        state_mean = ctrl_mean

    ctrl_norm  = ctrl_mean  / (np.linalg.norm(ctrl_mean)  + 1e-12)
    state_norm = state_mean / (np.linalg.norm(state_mean) + 1e-12)
    baseline_cos_sim = float(np.dot(ctrl_norm, state_norm))
    print(f"   [{state_label}] Baseline cos sim: {baseline_cos_sim:.4f}")

    # ── Auto-detect pre-computed shifts ───────────────────────────────────
    _sample_key = next(
        ((tid, ek) for (tid, ek) in state_dict.keys() if ek == "cell_emb"),
        None
    )
    if _sample_key:
        _raw = np.array(state_dict[_sample_key][:10], dtype=np.float64)
        _raw = _raw[np.isfinite(_raw)]
        _raw_med = float(np.median(_raw)) if len(_raw) else 0.0
        print(f"   [{state_label}] Raw sample: {np.round(_raw, 4).tolist()}")
        print(f"   [{state_label}] Raw median: {_raw_med:.4f} | "
              f"baseline: {baseline_cos_sim:.4f} | "
              f"shift after sub: {_raw_med - baseline_cos_sim:.5f}")
        if abs(_raw_med) < 0.1:
            print(f"   [{state_label}] Pre-computed shifts detected. "
                  f"Setting baseline=0.")
            baseline_cos_sim = 0.0

    # ── Extract per-gene shifts ───────────────────────────────────────────
    all_gene_shifts  = {}
    all_gene_n_cells = {}
    skipped_too_few  = 0

    for (token_id, emb_key), cos_sims in state_dict.items():
        if emb_key != "cell_emb":
            continue
        arr = np.array(cos_sims, dtype=np.float64)
        arr = arr[np.isfinite(arr)]
        all_gene_n_cells[token_id] = len(arr)
        if len(arr) < MIN_CELLS_PER_GENE:
            skipped_too_few += 1
            continue
        all_gene_shifts[token_id] = arr - baseline_cos_sim

    if skipped_too_few:
        print(f"   [{state_label}] {skipped_too_few} genes skipped "
              f"(< {MIN_CELLS_PER_GENE} cells).")

    candidate_gene_shifts = {
        tid: s for tid, s in all_gene_shifts.items()
        if tid in candidate_token_ids_set
    }
    other_gene_shifts = {
        tid: s for tid, s in all_gene_shifts.items()
        if tid not in candidate_token_ids_set
    }

    print(f"   [{state_label}] Candidate genes with data: "
          f"{len(candidate_gene_shifts)}/{len(candidate_token_ids_set)}")

    if candidate_gene_shifts:
        _all_s = np.concatenate(list(candidate_gene_shifts.values()))
        _all_s = _all_s[np.isfinite(_all_s)]
        print(f"   [{state_label}] Shift range: "
              f"[{_all_s.min():.5f}, {_all_s.max():.5f}], "
              f"median={np.median(_all_s):.5f}")

    if not candidate_gene_shifts and not candidate_token_ids_set:
        print(f"   [{state_label}] No candidate data — skipping.")
        return None

    # ── Random baseline pool ──────────────────────────────────────────────
    if other_gene_shifts:
        other_pool = np.concatenate(list(other_gene_shifts.values()))
        other_pool = other_pool[np.isfinite(other_pool)]
    else:
        other_pool = (np.concatenate(list(candidate_gene_shifts.values()))
                      if candidate_gene_shifts else np.zeros(10))
        other_pool = other_pool[np.isfinite(other_pool)]

    print(f"   [{state_label}] Baseline pool: n={len(other_pool)}, "
          f"median={np.median(other_pool):.5f}, std={np.std(other_pool):.5f}")

    # ── Score each candidate gene ─────────────────────────────────────────
    rows = []
    for token_id in candidate_token_ids_set:
        ensembl_id  = token_to_ensembl.get(token_id, str(token_id))
        match       = candidate_df.loc[
            candidate_df["ensembl_id"] == ensembl_id, "gene_symbol"
        ]
        gene_symbol = match.values[0] if len(match) > 0 else ensembl_id
        n_cells     = all_gene_n_cells.get(token_id, 0)

        if token_id in candidate_gene_shifts:
            shifts_a = candidate_gene_shifts[token_id]
            n_a      = len(shifts_a)

            other_for_this = np.concatenate([
                v for tid, v in other_gene_shifts.items()
                if tid != token_id
            ]) if other_gene_shifts else other_pool
            other_for_this = other_for_this[np.isfinite(other_for_this)]

            if len(other_for_this) >= n_a:
                sample_b = rng.choice(other_for_this, size=n_a, replace=False)
            elif len(other_for_this) > 0:
                sample_b = rng.choice(other_for_this, size=n_a, replace=True)
            else:
                sample_b = np.zeros(n_a)

            baseline_was_floored = False
            if np.median(sample_b) < 0:
                sample_b = np.maximum(sample_b, 0.0)
                baseline_was_floored = True

            try:
                _, pval = stats.ranksums(shifts_a, sample_b)
            except Exception:
                pval = 1.0
            if not np.isfinite(pval):
                pval = 1.0

            median_shift = float(np.nanmedian(shifts_a))
            mean_shift   = float(np.nanmean(shifts_a))
            std_shift    = float(np.nanstd(shifts_a))
        else:
            pval                 = 1.0
            baseline_was_floored = False
            median_shift = mean_shift = std_shift = 0.0

        rows.append({
            "gene_symbol":          gene_symbol,
            "ensembl_id":           ensembl_id,
            "median_cosine_shift":  median_shift,
            "mean_cosine_shift":    mean_shift,
            "std_cosine_shift":     std_shift,
            "median_cos_sim":       float(median_shift + baseline_cos_sim),
            "mean_cos_sim":         float(mean_shift   + baseline_cos_sim),
            "n_cells":              n_cells,
            "baseline_was_floored": baseline_was_floored,
            "pval_raw":             float(pval),
            "cell_type":            cell_type,
            "control_state":        CONTROL_LABEL,
            "target_state":         state_label,
            "perturb_mode":         PERTURB_MODE,
            "baseline_cos_sim":     baseline_cos_sim,
        })

    if not rows:
        return None

    df = pd.DataFrame(rows)
    df["median_cosine_shift"] = df["median_cosine_shift"].fillna(0.0)
    df["mean_cosine_shift"]   = df["mean_cosine_shift"].fillna(0.0)
    df["pval_raw"]            = df["pval_raw"].fillna(1.0)

    _, pval_adj, _, _ = multipletests(
        df["pval_raw"], alpha=0.05, method="fdr_bh"
    )
    df["pval_adj"] = pval_adj

    if gf_perturb_type == "delete":
        df["significant"] = (
            (df["pval_adj"] < 0.05) &
            (df["median_cosine_shift"].abs() > EFFECT_SIZE_THRESHOLD) &
            (df["median_cosine_shift"] < 0) &
            (df["n_cells"] >= MIN_CELLS_PER_GENE)
        )
    else:
        df["significant"] = (
            (df["pval_adj"] < 0.05) &
            (df["median_cosine_shift"].abs() > EFFECT_SIZE_THRESHOLD) &
            (df["median_cosine_shift"] > 0) &
            (df["n_cells"] >= MIN_CELLS_PER_GENE)
        )

    # Belt-and-braces: force non-significant for low-cell genes
    df.loc[df["n_cells"] < MIN_CELLS_PER_GENE, "significant"] = False

    ascending = (gf_perturb_type == "delete")
    df = df.sort_values("median_cosine_shift",
                        ascending=ascending).reset_index(drop=True)
    return df


# ── Main loop: all cell types × all states ────────────────────────────────────
for cell_type in candidate_dfs.keys():
    print(f"\n{'='*60}")
    print(f"Stats for: {cell_type}")

    candidate_df = candidate_dfs[cell_type]
    candidate_ensembl_set = set(candidate_df["ensembl_id"].dropna().tolist())
    candidate_token_ids_set = {
        tk.gene_token_dict[eid]
        for eid in candidate_ensembl_set
        if eid in tk.gene_token_dict
    }

    ct_isp_dir   = os.path.join(ISP_OUTPUT_DIR, cell_type.replace(" ", "_"))
    pickle_files = [
        f for f in os.listdir(ct_isp_dir)
        if f.endswith(".pickle") and "dict_cell_embs_" in f
    ]
    print(f"   {len(pickle_files)} pickle files found.")
    if not pickle_files:
        print("   No pickle files — skipping.")
        continue

    # Load all pickle files once — they contain all states simultaneously
    merged_by_state = defaultdict(lambda: defaultdict(list))
    n_loaded = n_failed = 0
    for fname in pickle_files:
        fpath = os.path.join(ct_isp_dir, fname)
        try:
            with open(fpath, "rb") as f:
                batch_dict = pickle.load(f)
            if isinstance(batch_dict, list):
                batch_dict = batch_dict[0] if batch_dict else {}
            for sl, inner_dict in batch_dict.items():
                for key, val in inner_dict.items():
                    if isinstance(val, list):
                        merged_by_state[sl][key].extend(val)
                    else:
                        merged_by_state[sl][key].append(val)
            n_loaded += 1
        except Exception as e:
            print(f"   Failed to load {fname}: {e}")
            n_failed += 1

    print(f"   Loaded {n_loaded} files ({n_failed} failed).")
    print(f"   States in pickles: {list(merged_by_state.keys())}")

    ct_stats_dir = os.path.join(STATS_OUTPUT_DIR, cell_type.replace(" ", "_"))
    os.makedirs(ct_stats_dir, exist_ok=True)

    # Score goal state and each alt state from the same pickle data
    for state_label in states_to_run:
        df_state = score_one_state(
            cell_type, state_label, candidate_df,
            candidate_token_ids_set, merged_by_state,
            embeddings, emb_feature_cols
        )
        if df_state is None:
            continue

        safe_state = state_label.replace(" ", "_")
        safe_ct    = cell_type.replace(" ", "_")
        out_csv    = os.path.join(
            ct_stats_dir,
            f"stats_{safe_ct}_{safe_state}.csv"
        )
        df_state.to_csv(out_csv, index=False)
        all_results.append(df_state)

        n_sig   = df_state["significant"].sum()
        n_total = len(df_state)
        print(f"   [{state_label}] {n_total} genes scored, "
              f"{n_sig} significant ({n_sig/n_total*100:.1f}%).")
        if n_sig > 0:
            top_cols = ["gene_symbol", "median_cosine_shift",
                        "mean_cosine_shift", "pval_adj", "n_cells"]
            print(df_state[df_state["significant"]][top_cols]
                  .head(10).to_string())

if all_results:
    final_df   = pd.concat(all_results, ignore_index=True)
    final_path = os.path.join(STATS_OUTPUT_DIR, "all_isp_results.csv")
    final_df.to_csv(final_path, index=False)
    print(f"\nAll results saved: {final_path}  ({len(final_df)} rows)")
    print(f"States in results: {final_df['target_state'].unique().tolist()}")
else:
    print("\nNo results to save.")

In [ ]:
import numpy as np
import pandas as pd
import pickle
import os
from collections import defaultdict
from scipy import stats
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

# ── Bootstrap stability check ─────────────────────────────────────────────────
# Resamples existing pickle data — no ISP rerun required.
# For each cell type and each state (goal + alt states):
#   - Splits per-gene shift arrays into two independent halves N_BOOTSTRAP_SPLITS times
#   - Computes top-10 hit overlap and Spearman rank correlation across halves
#   - Reports both metrics per cell type per state
# ─────────────────────────────────────────────────────────────────────────────

N_BOOTSTRAP_SPLITS = 10
SPLIT_FRACTION     = 0.5
MIN_CELLS_HALF     = 5

# States to assess stability for
_alt_states   = ALT_STATES if "ALT_STATES" in dir() and ALT_STATES else []
states_to_run = [TARGET_LABEL] + list(_alt_states)

stability_results = {}   # [cell_type][state_label] = metrics dict
rng_boot = np.random.default_rng(123)


def score_gene_shifts_stability(gene_shifts, other_pool, rng_obj,
                                effect_threshold=EFFECT_SIZE_THRESHOLD,
                                perturb_type='delete'):
    """
    Lightweight Liu et al. scorer for stability splits.
    Returns DataFrame with token_id, median_shift, significant columns.
    """
    rows = []
    for token_id, shifts_a in gene_shifts.items():
        if len(shifts_a) < MIN_CELLS_HALF:
            continue

        other_for_this = np.concatenate([
            v for tid, v in other_pool.items() if tid != token_id
        ]) if other_pool else np.zeros(len(shifts_a))
        other_for_this = other_for_this[np.isfinite(other_for_this)]

        if len(other_for_this) >= len(shifts_a):
            sample_b = rng_obj.choice(other_for_this,
                                      size=len(shifts_a), replace=False)
        elif len(other_for_this) > 0:
            sample_b = rng_obj.choice(other_for_this,
                                      size=len(shifts_a), replace=True)
        else:
            sample_b = np.zeros(len(shifts_a))

        if np.median(sample_b) < 0:
            sample_b = np.maximum(sample_b, 0.0)

        try:
            _, pval = stats.ranksums(shifts_a, sample_b)
        except Exception:
            pval = 1.0

        rows.append({
            'token_id':     token_id,
            'median_shift': float(np.nanmedian(shifts_a)),
            'pval_raw':     float(pval) if np.isfinite(pval) else 1.0,
        })

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    _, padj, _, _ = multipletests(
        df['pval_raw'].fillna(1.0), alpha=0.05, method='fdr_bh'
    )
    df['pval_adj'] = padj

    if perturb_type == 'delete':
        df['significant'] = (
            (df['pval_adj'] < 0.05) &
            (df['median_shift'].abs() > effect_threshold) &
            (df['median_shift'] < 0)
        )
    else:
        df['significant'] = (
            (df['pval_adj'] < 0.05) &
            (df['median_shift'].abs() > effect_threshold) &
            (df['median_shift'] > 0)
        )

    return df.sort_values('median_shift',
                          ascending=(perturb_type == 'delete'))


# ── Main stability loop ───────────────────────────────────────────────────────
for cell_type in candidate_dfs.keys():
    print(f"\n{'='*55}")
    print(f"Stability check: {cell_type}")

    ct_isp_dir   = os.path.join(ISP_OUTPUT_DIR, cell_type.replace(' ', '_'))
    pickle_files = [
        f for f in os.listdir(ct_isp_dir)
        if f.endswith('.pickle') and 'dict_cell_embs_' in f
    ]
    if not pickle_files:
        print("   No pickle files — skipping.")
        continue

    # ── Load all shift data per state per gene ────────────────────────────
    merged_by_state = defaultdict(lambda: defaultdict(list))
    n_loaded = n_failed = 0
    for fname in pickle_files:
        fpath = os.path.join(ct_isp_dir, fname)
        try:
            with open(fpath, 'rb') as f:
                batch_dict = pickle.load(f)
            if isinstance(batch_dict, list):
                batch_dict = batch_dict[0] if batch_dict else {}
            for state_label, inner_dict in batch_dict.items():
                for key, val in inner_dict.items():
                    if isinstance(val, list):
                        merged_by_state[state_label][key].extend(val)
                    else:
                        merged_by_state[state_label][key].append(val)
            n_loaded += 1
        except Exception as e:
            n_failed += 1

    print(f"   Loaded {n_loaded} pickle files ({n_failed} failed).")
    print(f"   States in pickles: {list(merged_by_state.keys())}")

    candidate_token_ids_set = {
        tk.gene_token_dict[eid]
        for eid in candidate_dfs[cell_type]['ensembl_id'].dropna()
        if eid in tk.gene_token_dict
    }

    stability_results[cell_type] = {}

    # ── Run stability check for each state ───────────────────────────────
    for state_label in states_to_run:
        if state_label not in merged_by_state:
            print(f"   [{state_label}] Not in pickles — skipping.")
            continue

        state_dict = merged_by_state[state_label]

        # Auto-detect pre-computed shifts vs raw cosine similarities
        _sample_key = next(
            ((tid, ek) for (tid, ek) in state_dict.keys()
             if ek == 'cell_emb'), None
        )
        baseline_cos_sim = 0.0
        if _sample_key:
            _raw = np.array(state_dict[_sample_key][:10], dtype=np.float64)
            _raw = _raw[np.isfinite(_raw)]
            _raw_med = float(np.median(_raw)) if len(_raw) else 0.0
            if abs(_raw_med) >= 0.1:
                # Raw cosine similarities — compute baseline from embeddings
                ctrl_mask  = ((embeddings['cell_type'] == cell_type) &
                              (embeddings['condition'] == CONTROL_LABEL))
                state_mask = ((embeddings['cell_type'] == cell_type) &
                              (embeddings['condition'] == state_label))
                if ctrl_mask.sum() > 0 and state_mask.sum() > 0:
                    ctrl_mean  = embeddings.loc[
                        ctrl_mask, emb_feature_cols].values.mean(axis=0)
                    state_mean = embeddings.loc[
                        state_mask, emb_feature_cols].values.mean(axis=0)
                    ctrl_norm  = ctrl_mean  / (np.linalg.norm(ctrl_mean)  + 1e-12)
                    state_norm = state_mean / (np.linalg.norm(state_mean) + 1e-12)
                    baseline_cos_sim = float(np.dot(ctrl_norm, state_norm))

        # Build per-gene shift arrays from full dataset
        gene_all_shifts = {}
        for (token_id, emb_key), cos_sims in state_dict.items():
            if emb_key != 'cell_emb':
                continue
            arr = np.array(cos_sims, dtype=np.float64)
            arr = arr[np.isfinite(arr)]
            if len(arr) >= MIN_CELLS_HALF * 2:
                gene_all_shifts[token_id] = arr - baseline_cos_sim

        candidate_gene_shifts = {
            tid: s for tid, s in gene_all_shifts.items()
            if tid in candidate_token_ids_set
        }
        other_gene_shifts = {
            tid: s for tid, s in gene_all_shifts.items()
            if tid not in candidate_token_ids_set
        }

        if len(candidate_gene_shifts) < 3:
            print(f"   [{state_label}] Fewer than 3 candidate genes "
                  f"with sufficient data — skipping.")
            continue

        print(f"   [{state_label}] {len(candidate_gene_shifts)} candidate "
              f"genes available for stability check.")

        # ── Bootstrap splits ──────────────────────────────────────────────
        top_hits_per_split = []
        spearman_per_split = []

        for split_i in range(N_BOOTSTRAP_SPLITS):
            half1_shifts = {}
            half2_shifts = {}

            for tid, arr in candidate_gene_shifts.items():
                n   = len(arr)
                idx = rng_boot.permutation(n)
                sp  = int(n * SPLIT_FRACTION)
                h1  = arr[idx[:sp]]
                h2  = arr[idx[sp:]]
                if len(h1) >= MIN_CELLS_HALF:
                    half1_shifts[tid] = h1
                if len(h2) >= MIN_CELLS_HALF:
                    half2_shifts[tid] = h2

            # Split other gene pool for baseline
            other_h1 = {}
            other_h2 = {}
            for tid, arr in other_gene_shifts.items():
                n   = len(arr)
                idx = rng_boot.permutation(n)
                sp  = int(n * SPLIT_FRACTION)
                h1  = arr[idx[:sp]]
                h2  = arr[idx[sp:]]
                if len(h1) >= MIN_CELLS_HALF:
                    other_h1[tid] = h1
                if len(h2) >= MIN_CELLS_HALF:
                    other_h2[tid] = h2

            df1 = score_gene_shifts_stability(
                half1_shifts, other_h1, rng_boot,
                perturb_type=gf_perturb_type
            )
            df2 = score_gene_shifts_stability(
                half2_shifts, other_h2, rng_boot,
                perturb_type=gf_perturb_type
            )

            if df1.empty or df2.empty:
                continue

            # ── Top-10 overlap ────────────────────────────────────────────
            top_n = min(10, len(df1), len(df2))
            top1  = set(df1.head(top_n)['token_id'].tolist())
            top2  = set(df2.head(top_n)['token_id'].tolist())
            top_hits_per_split.append(len(top1 & top2) / top_n)

            # ── Spearman rank correlation ──────────────────────────────────
            common_tids = set(df1['token_id']) & set(df2['token_id'])
            if len(common_tids) >= 5:
                df1_idx = df1.set_index('token_id')
                df2_idx = df2.set_index('token_id')
                common_list = list(common_tids)
                s1 = [df1_idx.loc[t, 'median_shift'] for t in common_list]
                s2 = [df2_idx.loc[t, 'median_shift'] for t in common_list]
                rho, _ = spearmanr(s1, s2)
                if np.isfinite(rho):
                    spearman_per_split.append(float(rho))

        if not top_hits_per_split:
            print(f"   [{state_label}] Insufficient data for stability.")
            continue

        mean_overlap = float(np.mean(top_hits_per_split))
        std_overlap  = float(np.std(top_hits_per_split))
        mean_rho     = float(np.mean(spearman_per_split)) \
                       if spearman_per_split else np.nan
        std_rho      = float(np.std(spearman_per_split)) \
                       if spearman_per_split else np.nan

        stability_results[cell_type][state_label] = {
            'mean_top10_overlap': mean_overlap,
            'std_top10_overlap':  std_overlap,
            'mean_spearman_rho':  mean_rho,
            'std_spearman_rho':   std_rho,
            'n_splits':           len(top_hits_per_split),
        }

        # Interpret overlap
        if mean_overlap >= 0.7:
            ov_interp = '✅ High'
        elif mean_overlap >= 0.5:
            ov_interp = '⚠️  Moderate'
        else:
            ov_interp = '❌ Low'

        # Interpret Spearman
        if np.isfinite(mean_rho):
            if mean_rho >= 0.7:
                rho_interp = '✅ High'
            elif mean_rho >= 0.5:
                rho_interp = '⚠️  Moderate'
            else:
                rho_interp = '❌ Low'
            rho_str = f"{mean_rho:.3f} ± {std_rho:.3f}  {rho_interp}"
        else:
            rho_str = 'N/A'

        print(f"   [{state_label}] "
              f"Top-10: {mean_overlap:.1%} ± {std_overlap:.1%}  {ov_interp}  |  "
              f"Spearman ρ: {rho_str}")


# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'='*75}")
print("STABILITY SUMMARY")
print('='*75)
print(f"{'Cell type':<28} {'State':<20} {'Top-10':>8} {'Spearman ρ':>14}")
print('-'*75)

all_stab_rows = []
for ct, state_dict_res in stability_results.items():
    for state_label, res in state_dict_res.items():
        rho_val = res.get('mean_spearman_rho', np.nan)
        rho_str = (f"{rho_val:.3f} ± {res['std_spearman_rho']:.3f}"
                   if np.isfinite(rho_val) else 'N/A')
        print(f"  {ct:<26}  {state_label:<20}  "
              f"{res['mean_top10_overlap']:.0%} ± "
              f"{res['std_top10_overlap']:.0%}   {rho_str}")
        all_stab_rows.append({
            'cell_type':          ct,
            'state':              state_label,
            'mean_top10_overlap': res['mean_top10_overlap'],
            'std_top10_overlap':  res['std_top10_overlap'],
            'mean_spearman_rho':  rho_val,
            'std_spearman_rho':   res.get('std_spearman_rho', np.nan),
            'n_splits':           res['n_splits'],
        })

# Save
stab_df  = pd.DataFrame(all_stab_rows)
stab_path = os.path.join(STATS_OUTPUT_DIR, 'stability_results.csv')
stab_df.to_csv(stab_path, index=False)
print(f"\n✅ Stability results saved: {stab_path}")

## 11. Rank & Summarize Results

In [ ]:
import numpy as np, pandas as pd
from statsmodels.stats.multitest import multipletests

results_df = None

if all_results:
    results_df = pd.concat(all_results, ignore_index=True)

    # ── NaN safety ────────────────────────────────────────────────────────────
    results_df = results_df.dropna(subset=["median_cosine_shift"])
    results_df["median_cosine_shift"] = results_df["median_cosine_shift"].fillna(0.0)
    results_df["mean_cosine_shift"]   = results_df["mean_cosine_shift"].fillna(0.0)
    results_df["pval_adj"]            = results_df["pval_adj"].fillna(1.0)

    # ── States present ────────────────────────────────────────────────────────
    _alt_states = ALT_STATES if "ALT_STATES" in dir() and ALT_STATES else []
    states_in_results = results_df["target_state"].unique().tolist() \
                        if "target_state" in results_df.columns \
                        else [TARGET_LABEL]
    print(f"States in results_df: {states_in_results}")

    # ── Summary: Goal state ───────────────────────────────────────────────────
    goal_df = results_df[
        results_df["target_state"] == TARGET_LABEL
    ] if "target_state" in results_df.columns else results_df

    print(f"\n{'='*60}")
    print(f"GOAL STATE SUMMARY: {CONTROL_LABEL} -> {TARGET_LABEL}")
    print(f"{'='*60}")
    print(f"Total rows: {len(goal_df)}")
    print(f"Significant hits: {goal_df['significant'].sum()}")
    print(f"\nTop hits by cell type:")

    for ct in passing_celltypes:
        ct_sig = goal_df[
            (goal_df["cell_type"] == ct) & (goal_df["significant"])
        ].sort_values("median_cosine_shift", ascending=(gf_perturb_type == "delete"))

        print(f"\n  {ct} ({len(ct_sig)} significant):")
        if len(ct_sig) > 0:
            print(ct_sig[["gene_symbol", "median_cosine_shift",
                           "mean_cosine_shift", "pval_adj", "n_cells"]].head(10).to_string())
        else:
            print("    No significant hits.")

    # ── Summary: each alt state ───────────────────────────────────────────────
    for alt_state in _alt_states:
        if alt_state not in states_in_results:
            print(f"\n  Alt state '{alt_state}' not in results — skipping.")
            continue

        alt_df = results_df[results_df["target_state"] == alt_state]

        print(f"\n{'='*60}")
        print(f"ALT STATE SUMMARY: {CONTROL_LABEL} -> {alt_state}")
        print(f"{'='*60}")
        print(f"Total rows: {len(alt_df)}")
        print(f"Significant hits: {alt_df['significant'].sum()}")
        print(f"\nTop hits by cell type:")

        for ct in passing_celltypes:
            ct_sig_alt = alt_df[
                (alt_df["cell_type"] == ct) & (alt_df["significant"])
            ].sort_values("median_cosine_shift",
                          ascending=(gf_perturb_type == "delete"))

            print(f"\n  {ct} ({len(ct_sig_alt)} significant):")
            if len(ct_sig_alt) > 0:
                print(ct_sig_alt[["gene_symbol", "median_cosine_shift",
                                   "mean_cosine_shift", "pval_adj",
                                   "n_cells"]].head(10).to_string())
            else:
                print("    No significant hits.")

    # ── Cross-state comparison ────────────────────────────────────────────────
    if _alt_states:
        print(f"\n{'='*60}")
        print("CROSS-STATE SPECIFICITY SUMMARY")
        print(f"{'='*60}")
        for ct in passing_celltypes:
            goal_sig_genes = set(
                goal_df[(goal_df["cell_type"] == ct) &
                        (goal_df["significant"])]["gene_symbol"].tolist()
            )
            for alt_state in _alt_states:
                if alt_state not in states_in_results:
                    continue
                alt_df_ct = results_df[
                    (results_df["target_state"] == alt_state) &
                    (results_df["cell_type"] == ct) &
                    (results_df["significant"])
                ]
                alt_sig_genes = set(alt_df_ct["gene_symbol"].tolist())
                goal_specific = goal_sig_genes - alt_sig_genes
                pan_disease   = goal_sig_genes & alt_sig_genes
                alt_specific  = alt_sig_genes  - goal_sig_genes

                print(f"\n  {ct}:")
                print(f"    {TARGET_LABEL}-specific : {sorted(goal_specific)}")
                print(f"    Pan-disease             : {sorted(pan_disease)}")
                print(f"    {alt_state}-specific    : {sorted(alt_specific)}")

else:
    print("No ISP results to summarize. Check earlier steps.")

## 12. Visualizations

In [ ]:
import numpy as np
import matplotlib
matplotlib.rcParams.update({
    "font.family": "sans-serif", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.8,
})
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd

ISP_FIG_DIR = os.path.join(OUTPUT_DIR, "isp_figures")
os.makedirs(ISP_FIG_DIR, exist_ok=True)

# ── Colour scheme ──────────────────────────────────────────────────────────────
SIG_COLOR           = "#C0392B"   # red  — goal state significant
NSIG_COLOR          = "#BDC3C7"   # grey — not significant
ALT_COLOR           = "#2980B9"   # blue — alt state significant
PAN_DISEASE_COLOR   = "#8E44AD"   # purple — significant in both states
GOAL_LIGHT          = "#E8A49C"   # light red — goal state, not significant
ALT_LIGHT           = "#A9C4D9"   # light blue — alt state, not significant

# ── Determine whether alternate states are available ──────────────────────────
# ALT_STATES should be defined in config cell, e.g. ALT_STATES = ["Cirrhosis"]
# If not defined, default to empty list so goal-state figures still work
_alt_states = ALT_STATES if "ALT_STATES" in dir() and ALT_STATES else []

if results_df is None or results_df.empty:
    print("No results to plot.")
else:
    shift_col = "median_cosine_shift"
    padj_col  = "pval_adj"
    gene_col  = "gene_symbol" if "gene_symbol" in results_df.columns else "ensembl_id"

    # ── Separate goal state and alt state results ──────────────────────────────
    goal_df = results_df[
        results_df.get("target_state", pd.Series([TARGET_LABEL]*len(results_df))) == TARGET_LABEL
    ].copy() if "target_state" in results_df.columns else results_df.copy()

    alt_dfs = {}
    for alt_state in _alt_states:
        if "target_state" in results_df.columns:
            _adf = results_df[results_df["target_state"] == alt_state].copy()
            if not _adf.empty:
                alt_dfs[alt_state] = _adf

    # ══════════════════════════════════════════════════════════════════════════
    # FIGURE 1 (existing): Control → Goal State
    # ══════════════════════════════════════════════════════════════════════════
    print("Generating Figure 1: Control → Goal State figures...")

    for ct in passing_celltypes:
        ct_df = goal_df[goal_df["cell_type"] == ct].copy()
        ct_df = ct_df.dropna(subset=[shift_col])
        if ct_df.empty:
            print(f"   No valid rows for {ct} — skipping.")
            continue

        ct_df["-log10_padj"] = -np.log10(ct_df[padj_col].clip(lower=1e-300))
        n_sig = ct_df["significant"].sum()

        fig, axes = plt.subplots(1, 3, figsize=(18, 6),
                                 gridspec_kw={"width_ratios": [2, 1.5, 1.5]})
        fig.suptitle(
            f"In-Silico Perturbation: {ct}\n"
            f"{CONTROL_LABEL} → {TARGET_LABEL}  |  "
            f"Mode: {PERTURB_MODE}-regulation  |  n={n_sig} significant hits",
            fontsize=12, fontweight="bold", y=1.02
        )

        # Panel A: Volcano
        ax = axes[0]
        colors_map = ct_df["significant"].map({True: SIG_COLOR, False: NSIG_COLOR})
        ax.scatter(ct_df[shift_col], ct_df["-log10_padj"],
                   c=colors_map, alpha=0.7, s=20, linewidths=0, rasterized=True)
        ax.axvline(0, color="black", lw=0.8, ls="--", alpha=0.6)
        ax.axhline(-np.log10(0.05), color="#7F8C8D", lw=0.8, ls=":", alpha=0.7)
        xlim = ax.get_xlim()
        ax.text(xlim[1], -np.log10(0.05) + 0.3, "FDR = 0.05",
                fontsize=7.5, ha="right", color="#7F8C8D")
        top_sig = ct_df[ct_df["significant"]].nsmallest(8, shift_col)
        for _, row in top_sig.iterrows():
            ax.annotate(str(row.get(gene_col, "")),
                        xy=(row[shift_col], row["-log10_padj"]),
                        xytext=(4, 4), textcoords="offset points", fontsize=7.5,
                        arrowprops=dict(arrowstyle="-", color="gray", lw=0.5))
        ax.set_xlabel("Median Cosine Shift (→ target state)", fontsize=11)
        ax.set_ylabel("-log₁₀(FDR-adjusted p-value)", fontsize=11)
        ax.set_title("A. Volcano Plot", fontsize=11, fontweight="bold", loc="left")
        ax.legend(handles=[
            mpatches.Patch(color=SIG_COLOR, label=f"Significant (n={n_sig})"),
            mpatches.Patch(color=NSIG_COLOR, label="Not significant"),
        ], fontsize=9, framealpha=0.8)

        # Panel B: Ranked bar chart
        ax2 = axes[1]
        top20 = ct_df.nsmallest(20, shift_col).copy()
        bar_colors = [SIG_COLOR if s else NSIG_COLOR for s in top20["significant"]]
        labels = top20[gene_col].tolist() if gene_col in top20.columns else [str(i) for i in top20.index]
        ax2.barh(range(len(top20)), top20[shift_col].values[::-1],
                 color=bar_colors[::-1], edgecolor="none", height=0.7)
        ax2.set_yticks(range(len(top20)))
        ax2.set_yticklabels(labels[::-1], fontsize=8)
        ax2.axvline(0, color="black", lw=0.8)
        ax2.set_xlabel("Median Cosine Shift", fontsize=10)
        ax2.set_title("B. Top 20 Perturbation Hits", fontsize=11, fontweight="bold", loc="left")

        # Panel C: Effect size distribution
        ax3 = axes[2]
        sig_shifts  = ct_df.loc[ct_df["significant"],  shift_col].dropna()
        nsig_shifts = ct_df.loc[~ct_df["significant"], shift_col].dropna()
        if len(sig_shifts) > 1 and sig_shifts.nunique() > 1:
            ax3.hist(sig_shifts.values, bins=min(20, len(sig_shifts)),
                     color=SIG_COLOR, alpha=0.7,
                     label=f"Significant (n={len(sig_shifts)})", density=True)
        if len(nsig_shifts) > 1 and nsig_shifts.nunique() > 1:
            ax3.hist(nsig_shifts.values, bins=min(20, len(nsig_shifts)),
                     color=NSIG_COLOR, alpha=0.5,
                     label=f"Not sig. (n={len(nsig_shifts)})", density=True)
        ax3.axvline(0, color="black", lw=0.8, ls="--", alpha=0.6)
        ax3.set_xlabel("Median Cosine Shift", fontsize=10)
        ax3.set_ylabel("Density", fontsize=10)
        ax3.set_title("C. Effect Size Distribution", fontsize=11, fontweight="bold", loc="left")
        ax3.legend(fontsize=8, framealpha=0.8)

        plt.tight_layout()
        fig_path = os.path.join(ISP_FIG_DIR, f"fig1_ctrl_vs_goal_{ct.replace(' ', '_')}.pdf")
        plt.savefig(fig_path, dpi=300, bbox_inches="tight", format="pdf")
        plt.savefig(fig_path.replace(".pdf", ".png"), dpi=200, bbox_inches="tight")
        plt.show()
        print(f"   Saved: {fig_path}")

    # ══════════════════════════════════════════════════════════════════════════
    # FIGURES 2–4 per cell type: Alt state figures (only if ALT_STATES defined)
    # ══════════════════════════════════════════════════════════════════════════
    for alt_state, alt_df in alt_dfs.items():

        print(f"\nGenerating alt state figures for: {alt_state}")

        for ct in passing_celltypes:

            # ── Shared data prep for this cell type ───────────────────────
            ct_goal = goal_df[goal_df["cell_type"] == ct].copy().dropna(subset=[shift_col])
            ct_alt  = alt_df[alt_df["cell_type"] == ct].copy().dropna(subset=[shift_col])

            if ct_goal.empty or ct_alt.empty:
                print(f"   Skipping {ct} — missing goal or alt state data.")
                continue

            goal_indexed = ct_goal.set_index(gene_col)
            alt_indexed  = ct_alt.set_index(gene_col)
            common_genes = goal_indexed.index.intersection(alt_indexed.index)

            if len(common_genes) < 3:
                print(f"   Skipping {ct} — fewer than 3 common genes.")
                continue

            # ── FIGURE 2: Control → Alt State ─────────────────────────────
            ct_alt["-log10_padj"] = -np.log10(ct_alt[padj_col].clip(lower=1e-300))
            n_sig_alt = ct_alt["significant"].sum()

            fig, axes = plt.subplots(1, 3, figsize=(18, 6),
                                     gridspec_kw={"width_ratios": [2, 1.5, 1.5]})
            fig.suptitle(
                f"In-Silico Perturbation: {ct}\n"
                f"{CONTROL_LABEL} → {alt_state}  |  "
                f"Mode: {PERTURB_MODE}-regulation  |  n={n_sig_alt} significant hits",
                fontsize=12, fontweight="bold", y=1.02
            )

            # Panel A: Volcano
            ax = axes[0]
            colors_alt = ct_alt["significant"].map({True: ALT_COLOR, False: NSIG_COLOR})
            ax.scatter(ct_alt[shift_col], ct_alt["-log10_padj"],
                       c=colors_alt, alpha=0.7, s=20, linewidths=0, rasterized=True)
            ax.axvline(0, color="black", lw=0.8, ls="--", alpha=0.6)
            ax.axhline(-np.log10(0.05), color="#7F8C8D", lw=0.8, ls=":", alpha=0.7)
            top_sig_alt = ct_alt[ct_alt["significant"]].nsmallest(8, shift_col)
            for _, row in top_sig_alt.iterrows():
                ax.annotate(str(row.get(gene_col, "")),
                            xy=(row[shift_col], row["-log10_padj"]),
                            xytext=(4, 4), textcoords="offset points", fontsize=7.5,
                            color=ALT_COLOR,
                            arrowprops=dict(arrowstyle="-", color="gray", lw=0.5))
            ax.set_xlabel("Median Cosine Shift (→ alt state)", fontsize=11)
            ax.set_ylabel("-log₁₀(FDR-adjusted p-value)", fontsize=11)
            ax.set_title("A. Volcano Plot", fontsize=11, fontweight="bold", loc="left")
            ax.legend(handles=[
                mpatches.Patch(color=ALT_COLOR, label=f"Significant (n={n_sig_alt})"),
                mpatches.Patch(color=NSIG_COLOR, label="Not significant"),
            ], fontsize=9, framealpha=0.8)

            # Panel B: Ranked bar chart
            ax2 = axes[1]
            top20_alt = ct_alt.nsmallest(20, shift_col).copy()
            bar_colors_alt = [ALT_COLOR if s else NSIG_COLOR for s in top20_alt["significant"]]
            labels_alt = top20_alt[gene_col].tolist()
            ax2.barh(range(len(top20_alt)), top20_alt[shift_col].values[::-1],
                     color=bar_colors_alt[::-1], edgecolor="none", height=0.7)
            ax2.set_yticks(range(len(top20_alt)))
            ax2.set_yticklabels(labels_alt[::-1], fontsize=8)
            ax2.axvline(0, color="black", lw=0.8)
            ax2.set_xlabel("Median Cosine Shift", fontsize=10)
            ax2.set_title(f"B. Top 20 — Control → {alt_state}",
                          fontsize=11, fontweight="bold", loc="left")

            # Panel C: Distribution
            ax3 = axes[2]
            sig_s_alt  = ct_alt.loc[ct_alt["significant"],  shift_col].dropna()
            nsig_s_alt = ct_alt.loc[~ct_alt["significant"], shift_col].dropna()
            if len(sig_s_alt) > 1 and sig_s_alt.nunique() > 1:
                ax3.hist(sig_s_alt.values, bins=min(20, len(sig_s_alt)),
                         color=ALT_COLOR, alpha=0.7,
                         label=f"Significant (n={len(sig_s_alt)})", density=True)
            if len(nsig_s_alt) > 1 and nsig_s_alt.nunique() > 1:
                ax3.hist(nsig_s_alt.values, bins=min(20, len(nsig_s_alt)),
                         color=NSIG_COLOR, alpha=0.5,
                         label=f"Not sig. (n={len(nsig_s_alt)})", density=True)
            ax3.axvline(0, color="black", lw=0.8, ls="--", alpha=0.6)
            ax3.set_xlabel("Median Cosine Shift", fontsize=10)
            ax3.set_ylabel("Density", fontsize=10)
            ax3.set_title("C. Effect Size Distribution",
                          fontsize=11, fontweight="bold", loc="left")
            ax3.legend(fontsize=8, framealpha=0.8)

            plt.tight_layout()
            fig2_path = os.path.join(
                ISP_FIG_DIR,
                f"fig2_ctrl_vs_alt_{ct.replace(' ', '_')}_{alt_state.replace(' ', '_')}.pdf"
            )
            plt.savefig(fig2_path, dpi=300, bbox_inches="tight", format="pdf")
            plt.savefig(fig2_path.replace(".pdf", ".png"), dpi=200, bbox_inches="tight")
            plt.show()
            print(f"   Saved Fig 2: {fig2_path}")

            # ── FIGURE 3: Goal State vs Alt State (differential) ───────────
            # x-axis = differential shift (goal - alt)
            # Negative = gene more important for goal state
            # Positive = gene more important for alt state
            goal_shifts = goal_indexed.loc[common_genes, shift_col]
            alt_shifts  = alt_indexed.loc[common_genes,  shift_col]
            diff_shifts = goal_shifts - alt_shifts

            sig_goal_common = goal_indexed.loc[common_genes, "significant"]
            sig_alt_common  = alt_indexed.loc[common_genes,  "significant"]

            goal_enriched = sig_goal_common & ~sig_alt_common
            alt_enriched  = ~sig_goal_common & sig_alt_common
            shared_sig    = sig_goal_common & sig_alt_common

            n_goal_enr = int(goal_enriched.sum())
            n_alt_enr  = int(alt_enriched.sum())
            n_shared   = int(shared_sig.sum())

            min_padj = np.minimum(
                goal_indexed.loc[common_genes, padj_col].fillna(1.0).values,
                alt_indexed.loc[common_genes,  padj_col].fillna(1.0).values
            )
            log_padj_diff = -np.log10(np.clip(min_padj, 1e-300, 1.0))

            colors_diff = np.where(
                goal_enriched, SIG_COLOR,
                np.where(
                    alt_enriched, ALT_COLOR,
                    np.where(shared_sig, PAN_DISEASE_COLOR, NSIG_COLOR)
                )
            )

            fig, axes = plt.subplots(1, 3, figsize=(18, 6),
                                     gridspec_kw={"width_ratios": [2, 1.5, 1.5]})
            fig.suptitle(
                f"ISP State Differential: {ct}\n"
                f"{TARGET_LABEL} vs {alt_state}  |  "
                f"{TARGET_LABEL}-enriched: {n_goal_enr}  |  "
                f"{alt_state}-enriched: {n_alt_enr}  |  "
                f"Shared: {n_shared}",
                fontsize=12, fontweight="bold", y=1.02
            )

            # Panel A: Differential volcano
            ax = axes[0]
            ax.scatter(diff_shifts.values, log_padj_diff,
                       c=colors_diff, alpha=0.75, s=20,
                       linewidths=0, rasterized=True)
            ax.axvline(0, color="black", lw=0.8, ls="--", alpha=0.6)
            ax.axhline(-np.log10(0.05), color="#7F8C8D", lw=0.8, ls=":", alpha=0.7)
            for cat_mask, cat_col_val, n_lab in [
                (goal_enriched, SIG_COLOR, 6),
                (alt_enriched,  ALT_COLOR, 6),
                (shared_sig,    PAN_DISEASE_COLOR, 3),
            ]:
                cat_genes = common_genes[cat_mask.values]
                if len(cat_genes) == 0:
                    continue
                top_diff = diff_shifts[cat_genes].abs().nlargest(n_lab).index
                for gene in top_diff:
                    gene_idx = list(common_genes).index(gene)
                    ax.annotate(
                        gene,
                        xy=(diff_shifts[gene], log_padj_diff[gene_idx]),
                        xytext=(4, 4), textcoords="offset points",
                        fontsize=7.5, color=cat_col_val,
                        arrowprops=dict(arrowstyle="-", color="gray", lw=0.4)
                    )
            ax.set_xlabel(
                f"← More important for {TARGET_LABEL}    "
                f"More important for {alt_state} →",
                fontsize=10
            )
            ax.set_ylabel("-log₁₀(min FDR p-value)", fontsize=11)
            ax.set_title(
                f"A. Differential Shift Volcano\n"
                f"({TARGET_LABEL} shift − {alt_state} shift)",
                fontsize=11, fontweight="bold", loc="left"
            )
            ax.legend(handles=[
                mpatches.Patch(color=SIG_COLOR,
                               label=f"{TARGET_LABEL}-enriched (n={n_goal_enr})"),
                mpatches.Patch(color=ALT_COLOR,
                               label=f"{alt_state}-enriched (n={n_alt_enr})"),
                mpatches.Patch(color=PAN_DISEASE_COLOR,
                               label=f"Shared (n={n_shared})"),
                mpatches.Patch(color=NSIG_COLOR, label="Not significant"),
            ], fontsize=8, framealpha=0.8)

            # Panel B: Top differentially shifted genes
            ax2 = axes[1]
            diff_sorted = diff_shifts.sort_values()
            n_show  = min(10, len(diff_sorted))
            top_neg = diff_sorted.head(n_show)
            top_pos = diff_sorted.tail(n_show).iloc[::-1]
            combined_diff = pd.concat([top_neg, top_pos])
            combined_diff = combined_diff[~combined_diff.index.duplicated()]
            bar_cols_diff = [SIG_COLOR if v < 0 else ALT_COLOR
                             for v in combined_diff.values]
            ax2.barh(range(len(combined_diff)),
                     combined_diff.values[::-1],
                     color=bar_cols_diff[::-1],
                     edgecolor="none", height=0.7)
            ax2.set_yticks(range(len(combined_diff)))
            ax2.set_yticklabels(combined_diff.index[::-1], fontsize=8)
            ax2.axvline(0, color="black", lw=0.8)
            ax2.set_xlabel("Differential Cosine Shift", fontsize=10)
            ax2.set_title(
                f"B. Top Differentially Shifted Genes\n"
                f"(neg = {TARGET_LABEL}-enriched, pos = {alt_state}-enriched)",
                fontsize=10, fontweight="bold", loc="left"
            )

            # Panel C: Paired bar for all significant genes
            ax3 = axes[2]
            all_sig_mask  = goal_enriched | alt_enriched | shared_sig
            all_sig_genes = common_genes[all_sig_mask.values]
            if len(all_sig_genes) > 0:
                ordered = pd.DataFrame({
                    "goal": goal_shifts[all_sig_genes].values,
                    "alt":  alt_shifts[all_sig_genes].values
                }, index=all_sig_genes).sort_values("goal")
                y_pos = np.arange(len(ordered))
                bar_h = 0.35
                ax3.barh(y_pos + bar_h/2, ordered["goal"].values[::-1],
                         height=bar_h, color=SIG_COLOR, alpha=0.85,
                         label=TARGET_LABEL, edgecolor="none")
                ax3.barh(y_pos - bar_h/2, ordered["alt"].values[::-1],
                         height=bar_h, color=ALT_COLOR, alpha=0.85,
                         label=alt_state, edgecolor="none")
                ax3.set_yticks(y_pos)
                ax3.set_yticklabels(ordered.index[::-1], fontsize=8)
                ax3.axvline(0, color="black", lw=0.8)
                ax3.set_xlabel("Median Cosine Shift", fontsize=10)
                ax3.set_title("C. Significant Genes: Paired Comparison",
                              fontsize=11, fontweight="bold", loc="left")
                ax3.legend(fontsize=9, framealpha=0.8)
            else:
                ax3.text(0.5, 0.5, "No significant genes\nin either state",
                         ha="center", va="center",
                         transform=ax3.transAxes, fontsize=12)
                ax3.set_title("C. Significant Genes: Paired Comparison",
                              fontsize=11, fontweight="bold", loc="left")

            plt.tight_layout()
            fig3_path = os.path.join(
                ISP_FIG_DIR,
                f"fig3_goal_vs_alt_{ct.replace(' ', '_')}_{alt_state.replace(' ', '_')}.pdf"
            )
            plt.savefig(fig3_path, dpi=300, bbox_inches="tight", format="pdf")
            plt.savefig(fig3_path.replace(".pdf", ".png"), dpi=200, bbox_inches="tight")
            plt.show()
            print(f"   Saved Fig 3: {fig3_path}")

            # ── FIGURE 4: Control, Goal and Alt States Together ────────────
            x_scatter = goal_shifts[common_genes]
            y_scatter = alt_shifts[common_genes]

            mash_specific = sig_goal_common & ~sig_alt_common
            pan_disease   = sig_goal_common & sig_alt_common
            alt_specific  = ~sig_goal_common & sig_alt_common

            n_mash_sp = int(mash_specific.sum())
            n_pan_d   = int(pan_disease.sum())
            n_alt_sp  = int(alt_specific.sum())

            color_scatter = np.where(
                mash_specific.values, SIG_COLOR,
                np.where(
                    pan_disease.values, PAN_DISEASE_COLOR,
                    np.where(alt_specific.values, ALT_COLOR, NSIG_COLOR)
                )
            )

            fig, axes = plt.subplots(
                1, 2, figsize=(16, 7),
                gridspec_kw={"width_ratios": [1.2, 1]}
            )
            fig.suptitle(
                f"ISP State Specificity: {ct}\n"
                f"Control → {TARGET_LABEL} vs Control → {alt_state}",
                fontsize=12, fontweight="bold", y=1.02
            )

            # Panel A: State specificity scatter
            ax = axes[0]
            ax.scatter(x_scatter, y_scatter,
                       c=color_scatter, alpha=0.75, s=30,
                       linewidths=0.3, edgecolors="white", rasterized=True)
            ax.axvline(0, color="black", lw=0.8, ls="--", alpha=0.5)
            ax.axhline(0, color="black", lw=0.8, ls="--", alpha=0.5)
            ax.axvline(-EFFECT_SIZE_THRESHOLD, color=SIG_COLOR,
                       lw=0.8, ls=":", alpha=0.35)
            ax.axhline(-EFFECT_SIZE_THRESHOLD, color=ALT_COLOR,
                       lw=0.8, ls=":", alpha=0.35)

            xlim_s = ax.get_xlim()
            ylim_s = ax.get_ylim()
            off = 0.04
            for txt, xf, yf, col, ha, va in [
                (f"{TARGET_LABEL}-specific\n(n={n_mash_sp})",
                 off, 1-off, SIG_COLOR, "left", "top"),
                (f"Pan-disease\n(n={n_pan_d})",
                 off, off, PAN_DISEASE_COLOR, "left", "bottom"),
                (f"{alt_state}-specific\n(n={n_alt_sp})",
                 1-off, off, ALT_COLOR, "right", "bottom"),
            ]:
                ax.text(
                    xlim_s[0] + (xlim_s[1]-xlim_s[0])*xf,
                    ylim_s[0] + (ylim_s[1]-ylim_s[0])*yf,
                    txt, fontsize=8, color=col, ha=ha, va=va,
                    bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                              alpha=0.7, edgecolor="none")
                )

            for cat_mask_s, cat_col_s, n_lab_s in [
                (mash_specific, SIG_COLOR, 6),
                (pan_disease,   PAN_DISEASE_COLOR, 5),
                (alt_specific,  ALT_COLOR, 4),
            ]:
                cat_genes_s = common_genes[cat_mask_s.values]
                if len(cat_genes_s) == 0:
                    continue
                combined_mag = (x_scatter[cat_genes_s].abs() +
                                y_scatter[cat_genes_s].abs())
                top_s = combined_mag.nlargest(n_lab_s).index
                for gene in top_s:
                    ax.annotate(
                        gene,
                        xy=(x_scatter[gene], y_scatter[gene]),
                        xytext=(5, 5), textcoords="offset points",
                        fontsize=7.5, color=cat_col_s,
                        arrowprops=dict(arrowstyle="-", color="gray", lw=0.4)
                    )

            ax.set_xlabel(f"Cosine shift toward {TARGET_LABEL}", fontsize=11)
            ax.set_ylabel(f"Cosine shift toward {alt_state}", fontsize=11)
            ax.set_title("A. State Specificity Scatter",
                         fontsize=11, fontweight="bold", loc="left")
            ax.legend(handles=[
                mpatches.Patch(color=SIG_COLOR,
                               label=f"{TARGET_LABEL}-specific (n={n_mash_sp})"),
                mpatches.Patch(color=PAN_DISEASE_COLOR,
                               label=f"Pan-disease (n={n_pan_d})"),
                mpatches.Patch(color=ALT_COLOR,
                               label=f"{alt_state}-specific (n={n_alt_sp})"),
                mpatches.Patch(color=NSIG_COLOR, label="Not significant"),
            ], fontsize=8, framealpha=0.8)

            # Panel B: Paired bar chart — top 15 goal hits with alt shifts
            ax2 = axes[1]
            top15_goal = goal_indexed.loc[common_genes].nsmallest(
                min(15, len(common_genes)), shift_col
            )
            top_genes_b = top15_goal.index.tolist()
            goal_vals_b = goal_indexed.loc[top_genes_b, shift_col]
            alt_vals_b  = alt_indexed.loc[top_genes_b,  shift_col]

            y_pos_b = np.arange(len(top_genes_b))
            bh = 0.35
            ax2.barh(
                y_pos_b + bh/2, goal_vals_b.values[::-1], height=bh,
                color=[SIG_COLOR if goal_indexed.loc[g, "significant"]
                       else GOAL_LIGHT for g in top_genes_b[::-1]],
                label=TARGET_LABEL, edgecolor="none"
            )
            ax2.barh(
                y_pos_b - bh/2, alt_vals_b.values[::-1], height=bh,
                color=[ALT_COLOR if alt_indexed.loc[g, "significant"]
                       else ALT_LIGHT for g in top_genes_b[::-1]],
                label=alt_state, edgecolor="none"
            )
            ax2.set_yticks(y_pos_b)
            ax2.set_yticklabels(top_genes_b[::-1], fontsize=8)
            ax2.axvline(0, color="black", lw=0.8)
            ax2.set_xlabel("Median Cosine Shift", fontsize=10)
            ax2.set_title(
                f"B. Top 15 Goal Hits: {TARGET_LABEL} vs {alt_state}\n"
                f"(saturated = significant, faded = not significant)",
                fontsize=10, fontweight="bold", loc="left"
            )
            ax2.legend(fontsize=9, framealpha=0.8)

            plt.tight_layout()
            fig4_path = os.path.join(
                ISP_FIG_DIR,
                f"fig4_three_state_{ct.replace(' ', '_')}_{alt_state.replace(' ', '_')}.pdf"
            )
            plt.savefig(fig4_path, dpi=300, bbox_inches="tight", format="pdf")
            plt.savefig(fig4_path.replace(".pdf", ".png"), dpi=200, bbox_inches="tight")
            plt.show()
            print(f"   Saved Fig 4: {fig4_path}")

    # # ══════════════════════════════════════════════════════════════════════════
    # # MULTI-CELL-TYPE DOT PLOT SUMMARY (goal state, unchanged)
    # # ══════════════════════════════════════════════════════════════════════════
    # if len(passing_celltypes) >= 2:
    #     all_sig = goal_df[goal_df["significant"]].dropna(subset=[shift_col]).copy()
    #     if len(all_sig) > 0:
    #         top_genes = (all_sig.groupby(gene_col)[shift_col].mean()
    #                      .nsmallest(20).index.tolist())
    #         pivot_shift = goal_df[goal_df[gene_col].isin(top_genes)].pivot_table(
    #             index=gene_col, columns="cell_type", values=shift_col, aggfunc="mean"
    #         ).fillna(0)
    #         pivot_padj = goal_df[goal_df[gene_col].isin(top_genes)].pivot_table(
    #             index=gene_col, columns="cell_type", values=padj_col, aggfunc="mean"
    #         ).fillna(1.0)

    #         x_labels = pivot_shift.columns.tolist()
    #         y_labels = pivot_shift.index.tolist()
    #         fig_dot, ax_dot = plt.subplots(
    #             figsize=(max(6, len(x_labels)*2.5), max(6, len(y_labels)*0.5))
    #         )
    #         for xi, ct in enumerate(x_labels):
    #             for yi, gene in enumerate(y_labels):
    #                 pv  = pivot_padj.loc[gene, ct] if ct in pivot_padj.columns else 1.0
    #                 sz  = max(20, -np.log10(pv + 1e-300) * 15)
    #                 col = SIG_COLOR if pv < 0.05 else NSIG_COLOR
    #                 ax_dot.scatter(xi, yi, s=sz, c=[col], alpha=0.85,
    #                                linewidths=0.5, edgecolors="black")
    #         ax_dot.set_xticks(range(len(x_labels)))
    #         ax_dot.set_xticklabels(x_labels, rotation=35, ha="right", fontsize=9)
    #         ax_dot.set_yticks(range(len(y_labels)))
    #         ax_dot.set_yticklabels(y_labels, fontsize=9)
    #         ax_dot.set_xlabel("Cell Type", fontsize=11)
    #         ax_dot.set_ylabel("Gene", fontsize=11)
    #         ax_dot.set_title(
    #             "Top Perturbation Hits Across Cell Types\n"
    #             "Dot size ∝ −log₁₀(FDR p-value)  |  Red = significant",
    #             fontsize=11, fontweight="bold"
    #         )
    #         ax_dot.grid(True, alpha=0.2, linewidth=0.5)
    #         ax_dot.set_xlim(-0.5, len(x_labels) - 0.5)
    #         ax_dot.set_ylim(-0.5, len(y_labels) - 0.5)
    #         plt.tight_layout()
    #         dot_path = os.path.join(ISP_FIG_DIR, "isp_dotplot_summary.pdf")
    #         plt.savefig(dot_path, dpi=300, bbox_inches="tight", format="pdf")
    #         plt.savefig(dot_path.replace(".pdf", ".png"), dpi=200, bbox_inches="tight")
    #         plt.show()
    #         print(f"   Saved: {dot_path}")
        # ══════════════════════════════════════════════════════════════════════════
    # MULTI-CELL-TYPE DOT PLOT: Control → Goal State (MASH)
    # ══════════════════════════════════════════════════════════════════════════
    print("\nGenerating dot plot: Control → Goal State across all cell types...")
    if len(passing_celltypes) >= 2:
        all_sig = goal_df[goal_df["significant"]].dropna(subset=[shift_col]).copy()
        if len(all_sig) > 0:
            top_genes = (all_sig.groupby(gene_col)[shift_col].mean()
                         .nsmallest(20).index.tolist())
            pivot_shift = goal_df[goal_df[gene_col].isin(top_genes)].pivot_table(
                index=gene_col, columns="cell_type", values=shift_col, aggfunc="mean"
            ).fillna(0)
            pivot_padj = goal_df[goal_df[gene_col].isin(top_genes)].pivot_table(
                index=gene_col, columns="cell_type", values=padj_col, aggfunc="mean"
            ).fillna(1.0)

            x_labels = pivot_shift.columns.tolist()
            y_labels = pivot_shift.index.tolist()
            fig_dot, ax_dot = plt.subplots(
                figsize=(max(6, len(x_labels)*2.5), max(6, len(y_labels)*0.5))
            )
            for xi, ct in enumerate(x_labels):
                for yi, gene in enumerate(y_labels):
                    pv  = pivot_padj.loc[gene, ct] if ct in pivot_padj.columns else 1.0
                    sz  = max(20, -np.log10(pv + 1e-300) * 15)
                    col = SIG_COLOR if pv < 0.05 else NSIG_COLOR
                    ax_dot.scatter(xi, yi, s=sz, c=[col], alpha=0.85,
                                   linewidths=0.5, edgecolors="black")
            ax_dot.set_xticks(range(len(x_labels)))
            ax_dot.set_xticklabels(x_labels, rotation=35, ha="right", fontsize=9)
            ax_dot.set_yticks(range(len(y_labels)))
            ax_dot.set_yticklabels(y_labels, fontsize=9)
            ax_dot.set_xlabel("Cell Type", fontsize=11)
            ax_dot.set_ylabel("Gene", fontsize=11)
            ax_dot.set_title(
                f"Top Perturbation Hits Across Cell Types — Control → {TARGET_LABEL}\n"
                "Dot size ∝ −log₁₀(FDR p-value)  |  Red = significant",
                fontsize=11, fontweight="bold"
            )
            ax_dot.grid(True, alpha=0.2, linewidth=0.5)
            ax_dot.set_xlim(-0.5, len(x_labels) - 0.5)
            ax_dot.set_ylim(-0.5, len(y_labels) - 0.5)
            plt.tight_layout()
            dot_path = os.path.join(ISP_FIG_DIR, "isp_dotplot_ctrl_vs_goal.pdf")
            plt.savefig(dot_path, dpi=300, bbox_inches="tight", format="pdf")
            plt.savefig(dot_path.replace(".pdf", ".png"), dpi=200, bbox_inches="tight")
            plt.show()
            print(f"   Saved: {dot_path}")

    # ══════════════════════════════════════════════════════════════════════════
    # MULTI-CELL-TYPE DOT PLOT: Control → Alt State
    # ══════════════════════════════════════════════════════════════════════════
    print("\nGenerating dot plot: Control → Alt State across all cell types...")
    for alt_state, alt_df in alt_dfs.items():
        if len(passing_celltypes) >= 2:
            all_sig_alt = alt_df[alt_df["significant"]].dropna(subset=[shift_col]).copy()
            if len(all_sig_alt) > 0:
                top_genes_alt = (all_sig_alt.groupby(gene_col)[shift_col].mean()
                                 .nsmallest(20).index.tolist())
                pivot_shift_alt = alt_df[alt_df[gene_col].isin(top_genes_alt)].pivot_table(
                    index=gene_col, columns="cell_type", values=shift_col, aggfunc="mean"
                ).fillna(0)
                pivot_padj_alt = alt_df[alt_df[gene_col].isin(top_genes_alt)].pivot_table(
                    index=gene_col, columns="cell_type", values=padj_col, aggfunc="mean"
                ).fillna(1.0)

                x_labels_alt = pivot_shift_alt.columns.tolist()
                y_labels_alt = pivot_shift_alt.index.tolist()
                fig_dot_alt, ax_dot_alt = plt.subplots(
                    figsize=(max(6, len(x_labels_alt)*2.5), max(6, len(y_labels_alt)*0.5))
                )
                for xi, ct in enumerate(x_labels_alt):
                    for yi, gene in enumerate(y_labels_alt):
                        pv  = pivot_padj_alt.loc[gene, ct] if ct in pivot_padj_alt.columns else 1.0
                        sz  = max(20, -np.log10(pv + 1e-300) * 15)
                        col = ALT_COLOR if pv < 0.05 else NSIG_COLOR
                        ax_dot_alt.scatter(xi, yi, s=sz, c=[col], alpha=0.85,
                                           linewidths=0.5, edgecolors="black")
                ax_dot_alt.set_xticks(range(len(x_labels_alt)))
                ax_dot_alt.set_xticklabels(x_labels_alt, rotation=35, ha="right", fontsize=9)
                ax_dot_alt.set_yticks(range(len(y_labels_alt)))
                ax_dot_alt.set_yticklabels(y_labels_alt, fontsize=9)
                ax_dot_alt.set_xlabel("Cell Type", fontsize=11)
                ax_dot_alt.set_ylabel("Gene", fontsize=11)
                ax_dot_alt.set_title(
                    f"Top Perturbation Hits Across Cell Types — Control → {alt_state}\n"
                    "Dot size ∝ −log₁₀(FDR p-value)  |  Blue = significant",
                    fontsize=11, fontweight="bold"
                )
                ax_dot_alt.grid(True, alpha=0.2, linewidth=0.5)
                ax_dot_alt.set_xlim(-0.5, len(x_labels_alt) - 0.5)
                ax_dot_alt.set_ylim(-0.5, len(y_labels_alt) - 0.5)
                plt.tight_layout()
                dot_alt_path = os.path.join(
                    ISP_FIG_DIR,
                    f"isp_dotplot_ctrl_vs_{alt_state.replace(' ', '_')}.pdf"
                )
                plt.savefig(dot_alt_path, dpi=300, bbox_inches="tight", format="pdf")
                plt.savefig(dot_alt_path.replace(".pdf", ".png"), dpi=200, bbox_inches="tight")
                plt.show()
                print(f"   Saved: {dot_alt_path}")
            else:
                print(f"   No significant alt state hits for {alt_state} — dot plot skipped.")

# ══════════════════════════════════════════════════════════════════════════
    # MASH vs alt TOP HITS COMPARISON — all cell types combined
    # Side-by-side ranked bar showing top hits in each state pooled across
    # all passing cell types, coloured by significance and state specificity.
    # Only genes significant in at least one state are included.
    # ══════════════════════════════════════════════════════════════════════════
    print("\nGenerating MASH vs MASLD top hits comparison across all cell types...")
    for alt_state, alt_df in alt_dfs.items():

        if goal_df.empty and alt_df.empty:
            print(f"   No data for either state — skipping comparison plot.")
            continue

        goal_all = goal_df.dropna(subset=[shift_col]).copy()
        alt_all  = alt_df.dropna(subset=[shift_col]).copy()

        if goal_all.empty and alt_all.empty:
            print(f"   No shift data — skipping.")
            continue

        # ── Genes significant in at least one state ───────────────────────
        goal_sig_genes = set(
            goal_df.loc[goal_df["significant"], gene_col].unique()
        )
        alt_sig_genes = set(
            alt_df.loc[alt_df["significant"], gene_col].unique()
        )

        # Union of significant genes across both states
        any_sig_genes = goal_sig_genes | alt_sig_genes

        if not any_sig_genes:
            print(f"   No significant genes in either state — skipping.")
            continue

        # ── Aggregate shifts from ALL rows with data ──────────────────────
        # Filter to only genes significant in at least one state
        goal_filt = goal_all[goal_all[gene_col].isin(any_sig_genes)]
        alt_filt  = alt_all[alt_all[gene_col].isin(any_sig_genes)]

        goal_agg = (goal_filt.groupby(gene_col)
                              .agg(
                                  goal_shift=(shift_col, "mean"),
                                  goal_padj=(padj_col,   "min"),
                              )
                              .reset_index())

        alt_agg  = (alt_filt.groupby(gene_col)
                             .agg(
                                 alt_shift=(shift_col, "mean"),
                                 alt_padj=(padj_col,   "min"),
                             )
                             .reset_index())

        # ── Breadth: number of cell types where gene is SIGNIFICANT ───────
        goal_nct = (goal_df[goal_df["significant"]]
                    .groupby(gene_col)["cell_type"]
                    .nunique()
                    .reset_index()
                    .rename(columns={"cell_type": "goal_n_celltypes"}))

        alt_nct  = (alt_df[alt_df["significant"]]
                    .groupby(gene_col)["cell_type"]
                    .nunique()
                    .reset_index()
                    .rename(columns={"cell_type": "alt_n_celltypes"}))

        # ── Select top genes from each state then take union ──────────────
        # Rank by most negative mean shift within significant genes only
        goal_sig_agg = goal_agg[goal_agg[gene_col].isin(goal_sig_genes)]
        alt_sig_agg  = alt_agg[alt_agg[gene_col].isin(alt_sig_genes)]

        top_goal = (goal_sig_agg.nsmallest(25, "goal_shift")[gene_col]
                    .tolist())
        top_alt  = (alt_sig_agg.nsmallest(25, "alt_shift")[gene_col]
                    .tolist())

        # Union preserving MASH-ranked order first then MASLD additions
        all_top_genes = list(dict.fromkeys(top_goal + top_alt))

        # ── Build compare_df ──────────────────────────────────────────────
        compare_df = pd.DataFrame({gene_col: all_top_genes})
        compare_df = compare_df.merge(goal_agg, on=gene_col, how="left")
        compare_df = compare_df.merge(alt_agg,  on=gene_col, how="left")
        compare_df = compare_df.merge(goal_nct, on=gene_col, how="left")
        compare_df = compare_df.merge(alt_nct,  on=gene_col, how="left")

        # Fill missing values — gene may be significant in one state only
        compare_df["goal_shift"]       = compare_df["goal_shift"].fillna(0.0)
        compare_df["alt_shift"]        = compare_df["alt_shift"].fillna(0.0)
        compare_df["goal_padj"]        = compare_df["goal_padj"].fillna(1.0)
        compare_df["alt_padj"]         = compare_df["alt_padj"].fillna(1.0)
        compare_df["goal_n_celltypes"] = (compare_df["goal_n_celltypes"]
                                          .fillna(0).astype(int))
        compare_df["alt_n_celltypes"]  = (compare_df["alt_n_celltypes"]
                                          .fillna(0).astype(int))

        # Significance flags
        compare_df["goal_sig"] = compare_df[gene_col].isin(goal_sig_genes)
        compare_df["alt_sig"]  = compare_df[gene_col].isin(alt_sig_genes)

        # Category for Panel C
        compare_df["category"] = np.where(
            compare_df["goal_sig"] & compare_df["alt_sig"],
            "Pan-disease",
            np.where(compare_df["goal_sig"],
                     f"{TARGET_LABEL}-specific",
            np.where(compare_df["alt_sig"],
                     f"{alt_state}-specific",
                     "Not significant"))
        )

        # Sort by MASH shift — consistent ordering across all panels
        compare_df = compare_df.sort_values("goal_shift").reset_index(drop=True)

        # ── Figure layout: 3 panels ───────────────────────────────────────
        n_genes = len(compare_df)
        fig_h   = max(6, n_genes * 0.45 + 2)

        fig_cmp, axes_cmp = plt.subplots(
            1, 3, figsize=(20, fig_h),
            gridspec_kw={"width_ratios": [1.5, 1.5, 1]}
        )
        fig_cmp.suptitle(
            f"Top Perturbation Hits: {TARGET_LABEL} vs {alt_state}\n"
            f"Pooled across all cell types  |  "
            f"Ranked by {TARGET_LABEL} cosine shift",
            fontsize=13, fontweight="bold", y=1.02
        )

        y_pos = np.arange(n_genes)
        bar_h = 0.38

        # Panel A: MASH shift
        ax_a = axes_cmp[0]
        goal_bar_colors = [
            SIG_COLOR if sig else GOAL_LIGHT
            for sig in compare_df["goal_sig"]
        ]
        ax_a.barh(y_pos, compare_df["goal_shift"].values[::-1],
                  height=bar_h * 1.8,
                  color=goal_bar_colors[::-1], edgecolor="none", alpha=0.9)
        ax_a.axvline(0, color="black", lw=0.9)
        ax_a.set_yticks(y_pos)
        ax_a.set_yticklabels(compare_df[gene_col].tolist()[::-1], fontsize=8.5)
        ax_a.set_xlabel("Mean Cosine Shift", fontsize=10)
        ax_a.set_title(
            f"A. {TARGET_LABEL}\n(saturated = significant)",
            fontsize=11, fontweight="bold", loc="left"
        )
        ax_a.legend(handles=[
            mpatches.Patch(color=SIG_COLOR,  label="Significant"),
            mpatches.Patch(color=GOAL_LIGHT, label="Not significant"),
        ], fontsize=8, loc="lower right", framealpha=0.8)
        ax_a.grid(True, axis="x", alpha=0.2, linewidth=0.5)

        # Panel B: alt state shift
        ax_b = axes_cmp[1]
        alt_bar_colors = [
            ALT_COLOR if sig else ALT_LIGHT
            for sig in compare_df["alt_sig"]
        ]
        ax_b.barh(y_pos, compare_df["alt_shift"].values[::-1],
                  height=bar_h * 1.8,
                  color=alt_bar_colors[::-1], edgecolor="none", alpha=0.9)
        ax_b.axvline(0, color="black", lw=0.9)
        ax_b.set_yticks(y_pos)
        ax_b.set_yticklabels([], fontsize=8.5)
        ax_b.set_xlabel("Mean Cosine Shift", fontsize=10)
        ax_b.set_title(
            f"B. {alt_state}\n(saturated = significant)",
            fontsize=11, fontweight="bold", loc="left"
        )
        ax_b.legend(handles=[
            mpatches.Patch(color=ALT_COLOR,  label="Significant"),
            mpatches.Patch(color=ALT_LIGHT,  label="Not significant"),
        ], fontsize=8, loc="lower right", framealpha=0.8)
        ax_b.grid(True, axis="x", alpha=0.2, linewidth=0.5)

        # Panel C: category dot + breadth scatter
        ax_c = axes_cmp[2]
        cat_palette = {
            f"{TARGET_LABEL}-specific": SIG_COLOR,
            f"{alt_state}-specific":    ALT_COLOR,
            "Pan-disease":              PAN_DISEASE_COLOR,
            "Not significant":          NSIG_COLOR,
        }
        gene_positions = {
            g: i for i, g in
            enumerate(compare_df[gene_col].tolist()[::-1])
        }
        for cat, col in cat_palette.items():
            mask = compare_df["category"] == cat
            if mask.sum() == 0:
                continue
            sub   = compare_df[mask]
            y_cat = [gene_positions[g] for g in sub[gene_col]]
            x_cat = (sub["goal_n_celltypes"] +
                     sub["alt_n_celltypes"]).values
            ax_c.scatter(x_cat, y_cat, c=[col]*len(y_cat), s=60,
                         alpha=0.85, linewidths=0.4, edgecolors="black",
                         label=cat, zorder=3)

        ax_c.set_yticks(y_pos)
        ax_c.set_yticklabels([], fontsize=8.5)
        ax_c.set_xlabel(
            "# cell types significant\n(MASH + MASLD combined)",
            fontsize=9
        )
        ax_c.set_title("C. Breadth\n(# cell types)",
                        fontsize=11, fontweight="bold", loc="left")
        ax_c.xaxis.set_major_locator(
            matplotlib.ticker.MaxNLocator(integer=True)
        )
        ax_c.legend(handles=[
            mpatches.Patch(color=c, label=cat)
            for cat, c in cat_palette.items()
            if (compare_df["category"] == cat).sum() > 0
        ], fontsize=7.5, loc="lower right", framealpha=0.8)
        ax_c.grid(True, alpha=0.2, linewidth=0.5)

        plt.tight_layout()
        cmp_path = os.path.join(
            ISP_FIG_DIR,
            f"isp_mash_vs_{alt_state.replace(' ', '_')}_all_celltypes.pdf"
        )
        plt.savefig(cmp_path, dpi=300, bbox_inches="tight", format="pdf")
        plt.savefig(cmp_path.replace(".pdf", ".png"), dpi=200,
                    bbox_inches="tight")
        plt.show()
        print(f"   Saved: {cmp_path}")

        # Save underlying data
        compare_df.to_csv(cmp_path.replace(".pdf", ".csv"), index=False)
        print(f"   Data saved: {cmp_path.replace('.pdf', '.csv')}")


print("\n✅ All ISP figures saved.")

## 13. Biological Validation — Pathway Enrichment (commented out code works for goal -> target e.g. MASH enrichment only, new code works for alt states as well e.g. MASLD)

In [ ]:
# import os, gseapy as gp

# if results_df is not None and not results_df.empty:
#     gene_label_col = 'gene_symbol' if 'gene_symbol' in results_df.columns else 'gene'
#     for ct in passing_celltypes:
#         ct_sig = results_df[(results_df['cell_type']==ct) & results_df['significant']]
#         sig_genes = ct_sig[gene_label_col].dropna().unique().tolist()
#         if len(sig_genes) < 5:
#             print(f'⚠️  {ct}: fewer than 5 significant genes, skipping enrichment.')
#             continue
#         print(f'\n Pathway enrichment for {ct} ({len(sig_genes)} genes)...')
#         try:
#             enr = gp.enrichr(gene_list=sig_genes, gene_sets=['KEGG_2021_Human','Reactome_2022','GO_Biological_Process_2023'], organism='human', outdir=os.path.join(OUTPUT_DIR, f'enrichment_{ct.replace(" ","_")}'), cutoff=0.05, no_plot=False)
#             top_paths = enr.results.nsmallest(10,'Adjusted P-value')[['Term','Adjusted P-value','Overlap','Genes']]
#             print(top_paths.to_string())
#             top_paths.to_csv(os.path.join(OUTPUT_DIR, f'enrichment_{ct.replace(" ","_")}_top10.csv'), index=False)
#         except Exception as e:
#             print(f'   ⚠️  Enrichment failed for {ct}: {e}')
# else:
#     print('No results available for enrichment.')

In [ ]:
import os, gseapy as gp

if results_df is not None and not results_df.empty:
    gene_label_col = 'gene_symbol' if 'gene_symbol' in results_df.columns else 'gene'

    _alt_states      = ALT_STATES if "ALT_STATES" in dir() and ALT_STATES else []
    states_to_enrich = [TARGET_LABEL] + list(_alt_states)

    # ── Per-state per-cell-type enrichment ───────────────────────────────────
    for state_label in states_to_enrich:
        print(f"\n{'='*55}")
        print(f"Pathway enrichment: {state_label}")

        if "target_state" in results_df.columns:
            state_df = results_df[
                results_df["target_state"] == state_label
            ].copy()
        else:
            state_df = results_df.copy()

        for ct in passing_celltypes:
            ct_sig    = state_df[
                (state_df['cell_type'] == ct) & state_df['significant']
            ]
            sig_genes = ct_sig[gene_label_col].dropna().unique().tolist()

            if len(sig_genes) < 5:
                print(f'   ⚠️  {ct} [{state_label}]: '
                      f'fewer than 5 significant genes — skipping.')
                continue

            print(f'\n    {ct} [{state_label}]: {len(sig_genes)} genes...')

            safe_ct    = ct.replace(' ', '_').replace('/', '_')
            safe_state = state_label.replace(' ', '_')
            outdir     = os.path.join(
                OUTPUT_DIR,
                f'enrichment_{safe_ct}_{safe_state}'
            )
            os.makedirs(outdir, exist_ok=True)

            try:
                enr = gp.enrichr(
                    gene_list=sig_genes,
                    gene_sets=['KEGG_2021_Human',
                               'Reactome_2022',
                               'GO_Biological_Process_2023'],
                    organism='human',
                    outdir=outdir,
                    cutoff=0.05,
                    no_plot=False
                )
                top_paths = enr.results.nsmallest(
                    10, 'Adjusted P-value'
                )[['Term', 'Adjusted P-value', 'Overlap', 'Genes']]
                print(top_paths.to_string())
                top_paths.to_csv(
                    os.path.join(
                        OUTPUT_DIR,
                        f'enrichment_{safe_ct}_{safe_state}_top10.csv'
                    ),
                    index=False
                )
            except Exception as e:
                print(f'   ⚠️  Enrichment failed for '
                      f'{ct} [{state_label}]: {e}')

    # ── Cross-state specificity enrichment ───────────────────────────────────
    if _alt_states and "target_state" in results_df.columns:
        print(f"\n{'='*55}")
        print("Cross-state specificity enrichment")

        for alt_state in _alt_states:
            for ct in passing_celltypes:
                goal_sig = set(
                    results_df[
                        (results_df['cell_type'] == ct) &
                        (results_df['target_state'] == TARGET_LABEL) &
                        results_df['significant']
                    ][gene_label_col].dropna().tolist()
                )
                alt_sig = set(
                    results_df[
                        (results_df['cell_type'] == ct) &
                        (results_df['target_state'] == alt_state) &
                        results_df['significant']
                    ][gene_label_col].dropna().tolist()
                )

                safe_ct    = ct.replace(' ', '_').replace('/', '_')
                safe_alt   = alt_state.replace(' ', '_')

                categories = {
                    f'{TARGET_LABEL}_specific': list(goal_sig - alt_sig),
                    f'{safe_alt}_specific':     list(alt_sig  - goal_sig),
                    'pan_disease':              list(goal_sig & alt_sig),
                }

                for cat_name, cat_genes in categories.items():
                    if len(cat_genes) < 5:
                        print(f'   ⚠️  {ct} [{cat_name}]: '
                              f'fewer than 5 genes — skipping.')
                        continue

                    print(f'\n    {ct} [{cat_name}]: '
                          f'{len(cat_genes)} genes...')

                    outdir = os.path.join(
                        OUTPUT_DIR,
                        f'enrichment_{safe_ct}_{cat_name}'
                    )
                    os.makedirs(outdir, exist_ok=True)

                    try:
                        enr = gp.enrichr(
                            gene_list=cat_genes,
                            gene_sets=['KEGG_2021_Human',
                                       'Reactome_2022',
                                       'GO_Biological_Process_2023'],
                            organism='human',
                            outdir=outdir,
                            cutoff=0.05,
                            no_plot=False
                        )
                        top_paths = enr.results.nsmallest(
                            10, 'Adjusted P-value'
                        )[['Term', 'Adjusted P-value', 'Overlap', 'Genes']]
                        print(top_paths.to_string())
                        top_paths.to_csv(
                            os.path.join(
                                OUTPUT_DIR,
                                f'enrichment_{safe_ct}_{cat_name}_top10.csv'
                            ),
                            index=False
                        )
                    except Exception as e:
                        print(f'   ⚠️  Enrichment failed for '
                              f'{ct} [{cat_name}]: {e}')

else:
    print('No results available for enrichment.')

## 14. Export Final Results

In [ ]:
import os, zipfile
from google.colab import files

zip_path = '/content/geneformer_isp_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk(OUTPUT_DIR):
        for fn in fnames:
            fp = os.path.join(root, fn)
            zf.write(fp, os.path.relpath(fp, '/content'))

zip_path = '/content/drive/MyDrive/scFMgeneformer_isp_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk(OUTPUT_DIR):
        for fn in fnames:
            fp = os.path.join(root, fn)
            zf.write(fp, os.path.relpath(fp, '/content'))

print(f'✅ All results zipped: {zip_path}')
files.download(zip_path)
print('\n Download triggered.')
print('\nKey output files:')
print('  geneformer_isp_results_full.csv  — full ranked gene table')
print('  candidate_genes.csv              — genes tested')
print('  isp_plot_*.png                   — volcano + bar plots per cell type')
print('  enrichment_*/                    — pathway enrichment results')
print('\n⚠️  INTERPRETATION REMINDER:')
print('  These results should be stated as:')
print('  "Geneformer predicts that perturbing gene X shifts [cell type] cell-state')
print('   representations toward [target state]."')
print('  NOT as causal claims without experimental Perturb-seq validation.')